In [ ]:
import os
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import re
import datetime
import statsmodels.api as sm
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

Add each fund sum asset col. calculate weight changes as deviation from revaluation /inflow-outflow change

# Loading Data

In [ ]:
# Y_funds_df = pd.read_csv(r'../../Data/Y/Y_shrinked_quart.csv', encoding='utf-16', sep='\t')
# # Y_funds_df.loc[Y_funds_df["Класс активов"].str.contains("foreign", case=False, na=False), "Класс активов"] = "stocks"
# Y_funds_df[~Y_funds_df['Company'].str.contains('рбк', case=False, na=False)]
# Y_funds_df['date'] = pd.to_datetime(Y_funds_df['date'])

In [ ]:
Y_funds_df = pd.read_csv(r'../../Data/Y/Y_interpolated.csv', encoding='utf-16', sep='\t')
# Y_funds_df.loc[Y_funds_df["Класс активов"].str.contains("foreign", case=False, na=False), "Класс активов"] = "stocks"
Y_funds_df[~Y_funds_df['Company'].str.contains('рбк', case=False, na=False)]
Y_funds_df['date'] = pd.to_datetime(Y_funds_df['date'])

In [ ]:
# Y_funds_df = pd.read_excel(r'../../Data/Y/Y.xlsx')
# # Y_funds_df.loc[Y_funds_df["Класс активов"].str.contains("foreign", case=False, na=False), "Класс активов"] = "stocks"
# Y_funds_df[~Y_funds_df['Company'].str.contains('рбк', case=False, na=False)]

In [ ]:
# Y_funds_df["total_assets"] = Y_funds_df.groupby(["date", "Название фонда"])["Сумма, руб."].transform("sum")/Y_funds_df.groupby(["date", "Название фонда"])["Доля, %"].transform("sum")
# Y_funds_df["classified_share"] = Y_funds_df.groupby(["date", "Название фонда"])["Доля, %"].transform("sum")

# # Compute the "other" row for each unique (date, Название фонда) group
# others = []

# # Group by date and Название фонда
# for (dt, fund), group in Y_funds_df.groupby(["date", "Название фонда"]):
#     # Calculate the remaining доля (%) and remaining Сумма, руб.
#     remaining_share = 1 - group["Доля, %"].sum()
#     remaining_sum = group["total_assets"].iloc[0] - group["Сумма, руб."].sum()
    
#     # Create a new row with minimal data as specified
#     new_row = {
#         "date": dt,
#         "Название фонда": fund,
#         "Класс активов": "other",
#         "Доля, %": remaining_share,
#         "Сумма, руб.": remaining_sum
#         # Other columns will be missing (NaN) by default.
#     }
#     others.append(new_row)

# # Convert the list of dictionaries to a DataFrame
# others_df = pd.DataFrame(others)

# # Append the "other" rows to the original dataframe without modifying the existing rows
# Y_funds_df = pd.concat([Y_funds_df, others_df], ignore_index=True)

In [ ]:
# # List of dates to drop
# dates_to_drop = [
#     "2021-08-31", "2021-07-30", "2021-05-31", "2021-04-30",
#     "2018-04-28", "2018-05-31", "2019-07-31", "2019-08-30",
#     "2019-01-31", "2020-01-31", "2019-05-31", "2019-10-31",
#     "2019-04-30", "2018-07-31", "2018-11-30", "2019-02-28",
#     "2018-10-31", "2019-11-29", "2021-02-26", "2020-05-29",
#     "2020-07-31", "2020-02-28", "2020-08-31", "2021-01-29",
#     "2020-04-30", "2020-10-30", "2020-11-30", "2018-08-31"
# ]

# # Ensure the date column is in datetime format
# Y_funds_df['date'] = pd.to_datetime(Y_funds_df['date'])

# # Convert the drop list to datetime
# dates_to_drop = pd.to_datetime(dates_to_drop)

# # Filter out rows where Y_funds_df['date'] is in the drop list
# Y_funds_df = Y_funds_df[~Y_funds_df['date'].isin(dates_to_drop)]

In [ ]:
# X_news_df = pd.read_excel(r'../../Data/X/News_panel_FBRT_adj.xlsx')
# X_news_df[~X_news_df['Company'].str.contains('рбк', case=False, na=False)]

In [ ]:
X_news_df = pd.read_excel(r'../../Data/X/News_panel_FBRT_adj_ru_class_interpalated.xlsx')
X_news_df[~X_news_df['Company'].str.contains('рбк', case=False, na=False)]

X_news_df['Negative'] = X_news_df['earnings_Negative']+X_news_df['forecast_Negative']+X_news_df['sanctions_Negative']+X_news_df['other_Negative']
X_news_df['Neutral'] = X_news_df['earnings_Neutral']+X_news_df['forecast_Neutral']+X_news_df['sanctions_Neutral']+X_news_df['other_Neutral']
X_news_df['Positive'] = X_news_df['earnings_Positive']+X_news_df['forecast_Positive']+X_news_df['sanctions_Positive']+X_news_df['other_Positive']

# Mapping X to Y - Creating Panel

In [ ]:
Panel = pd.merge(
    Y_funds_df,
    X_news_df.drop(
        columns=[col for col in X_news_df.columns 
                 if col in Y_funds_df.columns and col not in ['end_date', 'firm_in_text']]
    ),
    how='inner',
    left_on=['date', 'Company'],
    right_on=['end_date', 'firm_in_text']
)
Panel = Panel.drop_duplicates().dropna()

In [ ]:
# Panel = pd.merge(
#     Y_funds_df,
#     X_news_df.drop(
#         columns=[col for col in X_news_df.columns 
#                  if col in Y_funds_df.columns and col not in ['end_date', 'ISIN конечного актива']]
#     ),
#     how='inner',
#     left_on=['date', 'ISIN конечного актива'],
#     right_on=['end_date', 'ISIN конечного актива']
# )

# exceptions = ['date','start_date', 'end_date', 'Номер регистрации']

# Panel = Panel.apply(
#     lambda col: pd.to_numeric(col, errors='coerce').combine_sum(col) if col.name not in exceptions else col
# )
# Panel.dropna(inplace=True,subset='ISIN конечного актива')
# Panel.insert(1,'fund_key', Panel['Название фонда']+'__'+(Panel['Номер регистрации'].astype(str)))

In [ ]:
# sum merge on date and ISIN конечного актива
merge_isin = pd.merge(
    Y_funds_df,
    X_news_df.drop(
        columns=[col for col in X_news_df.columns 
                 if col in Y_funds_df.columns and col not in ['end_date', 'ISIN конечного актива']]
    ),
    how='inner',
    left_on=['date', 'ISIN конечного актива'],
    right_on=['end_date', 'ISIN конечного актива']
)

# Identify rows in Y_funds_df that did not match on ISIN конечного актива
matched_isin = merge_isin['ISIN конечного актива'].dropna().unique()
unmatched = Y_funds_df[~Y_funds_df['ISIN конечного актива'].isin(matched_isin)]

# Now merge these unmatched rows on date and company (for example, using 'Название фонда')
merge_company = pd.merge(
    unmatched,
    X_news_df.drop(
        columns=[col for col in X_news_df.columns 
                 if col in Y_funds_df.columns and col not in ['end_date', 'firm_in_text']]
    ),
    how='inner',
    left_on=['date', 'Company'],
    right_on=['end_date', 'firm_in_text']
)

# Combine the two merge results
Panel = pd.concat([merge_isin, merge_company], ignore_index=True).drop_duplicates()

try:
    Panel.insert(1,'fund_key', Panel['Название фонда']+'__'+(Panel['Номер регистрации'].astype(str)))
except: None

In [ ]:
Panel = Panel[~Panel['Название фонда'].str.contains('индекс', case=False, na=False)]

In [ ]:
# Panel.to_excel(r'../../Data/Panel.xlsx', index=False)

In [ ]:
total_assets_on_date = Panel.groupby(['date'])['Сумма, руб.'].sum().reset_index()
Panel.insert(12,'share_in_total_assets, %',0)
Panel['share_in_total_assets, %'] = (Panel['Сумма, руб.'] / Panel['date'].map(total_assets_on_date.set_index('date')['Сумма, руб.']))*100

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(12,6))
# plt.plot(total_assets_on_date['date'], total_assets_on_date['Сумма, руб.'], marker='o')
# plt.xlabel('Date')
# plt.ylabel('Сумма, руб.')
# plt.title('Total Assets Over Time')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

In [ ]:
Panel['Net_sent_score_partial'] = (Panel['Positive'] - Panel['Negative'])/(Panel['Positive'] + Panel['Negative'] + Panel['Neutral'])
Panel['Net_sent_score_weighted'] = Panel['Net_sent_score_partial'] * Panel['Доля, %']

Panel['Positive_weighted'] = Panel['Positive'] * Panel['Доля, %']
Panel['Negative_weighted'] = Panel['Negative'] * Panel['Доля, %']
Panel['Neutral_weighted'] = Panel['Neutral'] * Panel['Доля, %']

## Mapping controls

In [ ]:
pd.read_excel(r'../../Data/Controls/controls.xlsx')

In [ ]:
# Calculate unique funds per date
unique_funds_by_date = Panel.groupby('date')['fund_key'].nunique().reset_index()
unique_funds_by_date.rename(columns={'fund_key': 'unique_funds'}, inplace=True)

# Load and aggregate controls
controls = pd.read_excel(r'../../Data/Controls/controls.xlsx').reset_index(drop=True)
    # .drop([0, 1, 2], axis=0)\
    # .drop(columns=['date'])
controls = controls.groupby(by='date', as_index=False).agg({
    'Flow': 'sum',
    'Imoex': 'mean',
    'Ruonia': 'mean',
    'USD_RUB': 'mean',
    'RVI': 'mean'
})
controls['Flow'] = controls['Flow']/1000000000
# Ensure controls is sorted by date and compute imoex returns
controls = controls.sort_values('date').reset_index(drop=True)
controls["imoex_ret"] = controls["Imoex"].div(controls["Imoex"].shift(1)) - 1
controls["imoex_log_ret"] = np.log(controls["imoex_ret"])

total_assets_on_date.rename(columns={'Сумма, руб.': 'total_assets'}, inplace=True)
controls = controls.merge(total_assets_on_date, left_on='date', right_on='date', how='left')

# Merge the unique funds counts into controls if desired
controls = controls.merge(unique_funds_by_date, left_on='date', right_on='date', how='left')

# Add dummy: 0 for dates before March 2014, 1 for March 2014 and later
controls['dummy_mar2014'] = (pd.to_datetime(controls['date']) >= pd.Timestamp('2014-03-01')).astype(int)

# Add dummy: 0 for dates before February 2022, 1 for February 2022 and later
controls['dummy_feb2022'] = (pd.to_datetime(controls['date']) >= pd.Timestamp('2022-02-01')).astype(int)

controls['dummy_sep2018'] = (pd.to_datetime(controls['date']) >= pd.Timestamp('2018-09-01')).astype(int)

In [ ]:
Panel = pd.merge(Panel, controls, how='left', on='date')
Panel.drop_duplicates(inplace=True)
Panel.sort_values(by=['fund_key','firm_in_text', 'ISIN конечного актива','date'], inplace=True)

In [ ]:
Panel['RVI'].fillna(0, inplace=True)

In [ ]:
# Panel.to_excel(r'../../Data/Panels/Panel.xlsx', index=False)

# Aggregations

## Aggregation - i, t - firm

In [ ]:
Panel_agg_it = Panel.groupby(by=['firm_in_text', 'date'], as_index=False).agg({
    'Класс активов': 'sum',
    'Сумма, руб.': 'sum',
    'share_in_total_assets, %': 'sum',
    'Negative': 'sum',
    'Neutral': 'sum',
    'Positive': 'sum',
    'Flow':'sum',
    'Imoex':'sum',
    'Ruonia':'sum'
})

### Assets shares changes

In [ ]:
Panel_agg_it = Panel_agg_it.sort_values(by=["firm_in_text", "date"], ascending=[True, False])

Panel_agg_it.insert(5, "share_chng", Panel_agg_it.groupby("firm_in_text")["share_in_total_assets, %"].transform(
    lambda x: (x - x.shift(-1))
))
# Panel_agg_it.dropna(subset=['share_chng'], inplace=True)

In [ ]:
# Panel_agg_it.to_excel(r'../../Data/Panels/Agg_panel_i,t.xlsx', index=False)

## Aggregation - a, t - asset

In [ ]:
Panel_agg_at = Panel.groupby(by=['ISIN конечного актива', 'date'], as_index=False).agg({
    'Конечный актив': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
    'firm_in_text':'first',
    'Класс активов': 'first',
    'Сумма, руб.': 'sum',
    'share_in_total_assets, %': 'sum',
    'Negative': 'sum',
    'Neutral': 'sum',
    'Positive': 'sum',
    'Flow':'first',
    'Imoex':'first',
    'Ruonia':'first'
})

### Assets shares changes

In [ ]:
Panel_agg_at = Panel_agg_at.sort_values(by=["ISIN конечного актива", "date"], ascending=[True, False])

Panel_agg_at.insert(7, "share_chng", Panel_agg_at.groupby("ISIN конечного актива")["share_in_total_assets, %"].transform(
    lambda x: (x - x.shift(-1))
))
# Panel_agg_at.dropna(subset=['share_chng'], inplace=True)

In [ ]:
# Panel_agg_at.to_excel(r'../../Data/Panels/Agg_panel_a,t.xlsx', index=False)

## Aggregation - c, t - class

In [ ]:
# First, count the frequency of each fund_key grouped by date
fund_counts_by_date = Panel.groupby(['date', 'fund_key']).size().reset_index(name='count')

# Identify rows where a fund_key appears at least 10 times on a given date
valid_funds_by_date = fund_counts_by_date[fund_counts_by_date['count'] >= 10][['date', 'fund_key']]

# Filter the original DataFrame to keep only those rows
Panel_filtered = pd.merge(Panel, valid_funds_by_date, on=['date', 'fund_key'], how='inner')

Panel_filtered['total_assets'] = Panel_filtered.groupby('date')['Сумма, руб.'].transform('sum')

# Now perform the aggregation on the filtered DataFrame
Panel_agg_ct = Panel_filtered.groupby(by=['Класс активов', 'date'], as_index=False).agg({
    'Сумма, руб.': 'sum',
    'Flow': 'first',
    'Imoex': 'first',
    'imoex_ret': 'first',
    'Ruonia': 'first',
    'USD_RUB': 'first',
    'total_assets': 'first',
    'unique_funds': 'first',
    'dummy_mar2014': 'first',
    'dummy_feb2022': 'first'
})
Panel_agg_t = Panel.groupby(by=['date'], as_index=False).agg({
    'Negative': 'sum',
    'Neutral': 'sum',
    'Positive': 'sum'})

Panel_agg_ct = pd.merge(Panel_agg_ct, Panel_agg_t, how='left', on='date')

### Assets shares changes
Panel_agg_ct = Panel_agg_ct.sort_values(by=['Класс активов', 'date'], ascending=[True, False])

Panel_agg_ct.insert(6, "assets_class_share", Panel_agg_ct['Сумма, руб.'] / Panel_agg_ct['total_assets'])

Panel_agg_ct.insert(7, "share_chng", Panel_agg_ct.groupby(['Класс активов'])["assets_class_share"].transform(
    lambda x: (x - x.shift(-1))
))

## Aggregation - f, t

In [ ]:
Panel_agg_ft = Panel.groupby(by=['fund_key', 'date'], as_index=False).agg({
    'Класс активов': 'sum',
    'Сумма, руб.': 'sum',
    'Доля, %': 'sum',
    'Net_sent_score_weighted': 'sum',
    'Positive_weighted': 'sum',
    'Negative_weighted': 'sum',
    'Neutral_weighted': 'sum',
    'Negative': 'sum',
    'Neutral': 'sum',
    'Positive': 'sum',
    'Flow':'first',
    'Imoex':'first',
    'imoex_ret': 'first',
    'Ruonia':'first',
    'USD_RUB':'first',
    'RVI':'first',
    'unique_funds':'first',
    'dummy_mar2014':'first',
    'dummy_feb2022':'first'
})

## Aggregation - f, t, c with news classification

In [ ]:
Panel_agg_ftc = Panel.groupby(['fund_key', 'date'], as_index=False).agg({
    'Класс активов': 'sum',
    'Сумма, руб.': 'sum',
    'share_in_total_assets, %': 'sum',
    'Negative': 'sum',
    'Neutral':  'sum',
    'Positive': 'sum',
    'forecast_Negative': 'sum',
    'forecast_Neutral':  'sum',
    'forecast_Positive': 'sum',
    'earnings_Negative':'sum',
    'earnings_Neutral': 'sum',
    'earnings_Positive':'sum',
    'sanctions_Negative':'sum',
    'sanctions_Neutral':'sum',
    'sanctions_Positive':'sum',
    'other_Negative':   'sum',
    'other_Neutral':    'sum',
    'other_Positive':   'sum',
    'Flow': 'first',
    'Imoex': 'first',
    'imoex_ret': 'first',
    'Ruonia': 'first',
    'USD_RUB': 'first',
    'unique_funds': 'first',
    'dummy_mar2014': 'first',
    'dummy_feb2022': 'first'
})


In [ ]:
Panel.columns

## Aggregation - f, i, t

In [ ]:
Panel_agg_fit = Panel.groupby(by=['fund_key','firm_in_text', 'date'], as_index=False).agg({
    'Класс активов': 'first',
    'Сумма, руб.': 'sum',
    'share_in_total_assets, %': 'sum',
    'Negative': 'first',
    'Neutral': 'first',
    'Positive': 'first',
    'Flow':'first',
    'Imoex':'first',
    'Ruonia':'first',
    'USD_RUB':'first',
    'unique_funds':'first',
    'dummy_mar2014':'first',
    'dummy_feb2022':'first'
})

### Assets shares changes

In [ ]:
Panel_agg_fit = Panel_agg_fit.sort_values(by=['fund_key','firm_in_text', 'date'], ascending=[True ,True, False])

Panel_agg_fit.insert(6, "share_chng", Panel_agg_fit.groupby(['fund_key','firm_in_text'])["share_in_total_assets, %"].transform(
    lambda x: (x - x.shift(-1))
))

Panel_agg_fit.insert(6, "sum_chng", Panel_agg_fit.groupby(['fund_key','firm_in_text'])["Сумма, руб."].transform(
    lambda x: (x - x.shift(-1))
))

u = Panel_agg_fit[['date', 'Imoex']].drop_duplicates('date').sort_values('date')
u['imoex_ret'] = u['Imoex'].div(u['Imoex'].shift(1)) - 1

Panel_agg_fit = Panel_agg_fit.merge(u[['date', 'imoex_ret']], on='date', how='left')

# Panel_agg_fit.dropna(subset=['share_chng'], inplace=True)

In [ ]:
Panel_agg_fit.sort_values(by=['date', 'firm_in_text', 'fund_key'], ascending=[False, True, False], inplace=True)
Panel_agg_fit.drop(columns='weights_std', errors='ignore', inplace=True)
Panel_agg_fit.insert(7,'weights_std',Panel_agg_fit.groupby(['firm_in_text', 'date'])['share_chng'].transform('std'))
Panel_agg_fit.ffill(inplace=True)

In [ ]:
# Panel_agg_fit.to_excel(r'../../Data/Panels/Agg_panel_fi,t.xlsx', index=False)

# Calculation

## Portfolio volatility

In [ ]:
All_instr_df = pd.read_csv(r'../../Data/quotes_data/All_instr_df.csv', encoding = 'utf-16', sep = '\t')
All_instr_df['ISIN'].fillna(All_instr_df['SECID'], inplace=True)

portfolios = Panel[['date', 'fund_key', 'ISIN конечного актива', 'Доля, %']]
portfolios.sort_values(by=['date', 'fund_key', 'Доля, %'], ascending=[False, True, False], inplace= True)

In [ ]:
import pandas as pd
from joblib import Parallel, delayed

# 1) Pre‑index the market data *once* (just the one column we need)
mdf_indexed = (
    All_instr_df
    .loc[:, ['ISIN', 'TRADEDATE', 'ret']]
    .assign(TRADEDATE=lambda df: pd.to_datetime(df['TRADEDATE']))
    .set_index(['ISIN', 'TRADEDATE'])
    .sort_index()
)['ret']  # this is now a pd.Series with a MultiIndex

def attach_price_series_parallel(portfolios_df, price_series, N, n_jobs=-1):
    """
    For each row in portfolios_df, grab the last N days of
    price_series (ret) for that ISIN and date.
    
    Parameters
    ----------
    portfolios_df : pd.DataFrame
        Must have columns 'date' and 'ISIN конечного актива'
    price_series : pd.Series
        Market ret indexed by (ISIN, TRADEDATE)
    N : int
        Lookback window in days
    n_jobs : int
        Number of parallel workers
    
    Returns
    -------
    pd.DataFrame
        portfolios_df with new column 'price_series'
    """
    dfpf = portfolios_df.copy()
    dfpf['date'] = pd.to_datetime(dfpf['date'])
    
    # Prepare the list of tasks
    tasks = list(zip(dfpf['ISIN конечного актива'], dfpf['date']))

    def _get_ser(isin, dt):
        start = dt - pd.Timedelta(days=N-1)
        # try slicing, if ISIN absent return empty
        try:
            ser = price_series.loc[isin].loc[start:dt]
        except KeyError:
            return pd.Series(dtype=float)
        return ser.rename(isin)

    # Parallel extraction
    dfpf['price_series'] = Parallel(n_jobs=n_jobs)(
        delayed(_get_ser)(isin, dt) for isin, dt in tasks
    )
    return dfpf

# 2) Call it once, without copying All_instr_df repeatedly
out = attach_price_series_parallel(
    portfolios_df = portfolios.loc[:],
    price_series  = mdf_indexed,
    N             = 90,
    n_jobs        = 1    # or -1 to use all cores
)


In [ ]:
import pandas as pd

rows = []
for (dt, fk), grp in out.groupby(['date', 'fund_key']):
    # 1) Dedupe any repeated ISIN rows
    uniq = grp.drop_duplicates('ISIN конечного актива')
    isins = uniq['ISIN конечного актива'].tolist()
    
    # 2) Build weights Series
    weights = pd.Series(
        data=uniq['Доля, %'].values,
        index=isins,
        name='weights'
    )
    
    # 3) Build return‑matrix, dropping duplicate dates in each series
    ret_matrix = pd.DataFrame({
        isin: (ser[~ser.index.duplicated(keep='first')].rename(isin))
        for isin, ser in zip(isins, uniq['price_series'])
    })
    
    rows.append({
        'date':       dt,
        'fund_key':   fk,
        'weights':    weights,
        'ret_matrix': ret_matrix
    })

panel_df = pd.DataFrame(rows)

In [ ]:
# Initialize new columns
panel_df['cov_matrix']  = None
panel_df['weights_adj'] = None

# Loop and compute
for idx, row in panel_df.iterrows():
    # drop assets with no returns
    rm = row['ret_matrix'].dropna(axis=1, how='any')
    # adjusted weights
    w_adj = row['weights'].loc[rm.columns]
    # variance–covariance
    cov = rm.cov()
    # assign back
    panel_df.at[idx, 'weights_adj'] = w_adj
    panel_df.at[idx, 'cov_matrix']  = cov


In [ ]:
panel_df['portfolio_var'] = None
panel_df['portfolio_var'] = panel_df.apply(
    lambda row: row['weights_adj'].T @ row['cov_matrix'] @ row['weights_adj'],
    axis=1
)

In [ ]:
# panel_df.to_excel('portfolio_variances.xlsx', index=False)

## Amihud illiq

In [ ]:
import numpy as np

All_instr_df['TRADEDATE'] = pd.to_datetime(All_instr_df['TRADEDATE'])
All_instr_df['Amihud_illiq'] = All_instr_df['ret'].abs() / All_instr_df['VALUE']
All_instr_df.loc[All_instr_df['VALUE'] == 0, 'Amihud_illiq'] = np.nan
mdf_illiq = (All_instr_df
    .set_index(['ISIN', 'TRADEDATE'])
    .sort_index()['Amihud_illiq']
)

# Precompute set of ISINs present in the illiquidity series
available_isins = set(mdf_illiq.index.get_level_values(0))

N = 90
fund_liqs = []
for idx, row in panel_df.iterrows():
    dt    = row['date']
    start = dt - pd.Timedelta(days=N-1)
    w     = row['weights']
    
    # for each ISIN, compute mean illiquidity if data exists
    means = {}
    for isin, weight in w.items():
        if isin in available_isins:
            ser = mdf_illiq.loc[isin].loc[start:dt]
            if not ser.empty:
                means[isin] = ser.mean()
    if means:
        # align weights to available ISINs only
        w_adj = w.loc[means.keys()]
        fund_liqs.append((w_adj * pd.Series(means)).sum())
    else:
        fund_liqs.append(np.nan)

panel_df['fund_liquidity'] = fund_liqs


# NEW HYPOTHESES TESTING

In [ ]:
txt = results.summary.as_text()

fig = plt.figure(figsize=(7, len(txt.splitlines())*0.1))
fig.text(0, 1, txt, va='top', family='Lucida Console')
plt.axis('off')
plt.tight_layout()

In [ ]:
txt = re_res.summary.as_text()

fig = plt.figure(figsize=(7, len(txt.splitlines())*0.1))
fig.text(0, 1, txt, va='top', family='Lucida Console')
plt.axis('off')
plt.tight_layout()

## H1

In [ ]:
import pandas as pd
import math

def first_two_nonzero_round(x):
    """
    Round x so that the first two non-zero digits after the decimal are kept.
    """
    if pd.isna(x):
        return x
    absx = abs(x)
    if absx == 0:
        return 0.0
    frac = absx if absx < 1 else absx - math.floor(absx)
    if frac == 0:
        decimals = 0
    else:
        first_pos = math.ceil(-math.log10(frac))
        decimals = first_pos + 1
    return round(x, decimals)

def results_to_df_two_nonzero(results):
    """
    Convert PanelOLS results into a DataFrame with coefficients,
    standard errors, t-stats, p-values, two-digit dynamic rounding,
    significance stars, and a combined 'coef_sig' column.
    """
    df = pd.DataFrame({
        'coef':    results.params,
        'std_err': results.std_errors,
        't_stat':  results.tstats,
        'p_value': results.pvalues
    })
    # Apply dynamic two-digit rounding
    for col in ['coef', 'std_err', 't_stat', 'p_value']:
        df[col] = df[col].apply(first_two_nonzero_round)
    # Add significance stars
    def stars(p):
        if p < 0.01: return '***'
        elif p < 0.05: return '**'
        elif p < 0.10: return '*'
        return ''
    df['signif'] = df['p_value'].apply(stars)
    # Create combined coefficient + stars column
    df['coef_sig'] = df['coef'].astype(str) + df['signif']
    return df

# Example usage:
# results_df = results_to_df_two_nonzero(results)
# ace_tools.display_dataframe_to_user("FE Results with Coef+Stars", results_df)


In [ ]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

def run_panel_models(
    df,                 
    entity_col,         
    y_var,              
    main_vars,          
    controls,           
    lag=1,
    entity_effects=False,
    time_effects=False,
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
):
    # 1) prepare the data
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index([entity_col, 'date']).sort_index()
    # 2) compute lags of main_vars
    lagged = []
    for v in main_vars:
        lv = f"{v}_L{lag}"
        df[lv] = df.groupby(level=entity_col)[v].shift(lag)
        lagged.append(lv)
    df = df.dropna(subset=lagged)
    # 3) define three specifications
    specs = {
        'current': main_vars + controls,
        'lagged': lagged + controls,
        'both':   main_vars + lagged + controls
    }
    results = {}
    # 4) fit each spec
    for name, vars_ in specs.items():
        exog = sm.add_constant(df[vars_])
        model = PanelOLS(
            df[y_var],
            exog,
            entity_effects=entity_effects,
            time_effects=time_effects,
            drop_absorbed=True
        )
        results[name] = model.fit(
            cov_type=cov_type,
            cluster_entity=cluster_entity,
            cluster_time=cluster_time
        )
    return results

def panel_results_to_df(models):
    """
    Convert a dict of PanelOLS results into a single DataFrame.
    `models` should be a dict: {spec_name: fitted_result}.
    """
    rows = []
    for spec_name, res in models.items():
        # summary stats
        summary = {
            'R2_within':  res.rsquared_within,
            'R2_between': res.rsquared_between,
            'R2_overall': res.rsquared_overall,
            'N_obs':      int(res.nobs)
        }
        for term, val in summary.items():
            rows.append({
                'spec':     spec_name,
                'term':     term,
                'coef':     val,
                'std_err':  None,
                't_stat':   None,
                'p_value':  None,
                'signif':   '',
                'coef_sig': f"{val:.4f}"
            })

        # coefficient table
        coef_df = results_to_df_two_nonzero(res).reset_index().rename(columns={'index':'term'})
        for _, r in coef_df.iterrows():
            rows.append({
                'spec':     spec_name,
                'term':     r['term'],
                'coef':     r['coef'],
                'std_err':  r['std_err'],
                't_stat':   r['t_stat'],
                'p_value':  r['p_value'],
                'signif':   r['signif'],
                'coef_sig': r['coef_sig']
            })

    return pd.DataFrame(rows)

In [ ]:
Panel_H1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H1['portfolio_vol'] = np.sqrt(Panel_H1['portfolio_var'])

Panel_H1['net_sent_score'] = (Panel_H1['Positive'] - Panel_H1['Negative'])/(Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Neg_share'] = Panel_H1['Negative'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Pos_share'] = Panel_H1['Positive'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])

Panel_H1['Neg_share_weighted'] = Panel_H1['Negative_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])
Panel_H1['Pos_share_weighted'] = Panel_H1['Positive_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])

Panel_H1['log_portfolio_var'] = np.log(Panel_H1['portfolio_var'])
Panel_H1.loc[Panel_H1['portfolio_var'] == 0, 'log_portfolio_var'] = 0
Panel_H1 = Panel_H1[Panel_H1['portfolio_var'] > 0]

Panel_H1['RVI'].ffill(inplace=True)


# Panel_H1 = Panel_H1[(Panel_H1['net_sent_score'] >= Panel_H1['net_sent_score'].quantile(0.01)) &
#                     (Panel_H1['net_sent_score'] <= Panel_H1['net_sent_score'].quantile(0.99))]

# Panel_H1 = Panel_H1[(Panel_H1['Net_sent_score_weighted'] >= Panel_H1['Net_sent_score_weighted'].quantile(0.01)) &
#                     (Panel_H1['Net_sent_score_weighted'] <= Panel_H1['Net_sent_score_weighted'].quantile(0.99))]


# lower = Panel_H1['net_sent_score'].quantile(0.01)
# upper = Panel_H1['net_sent_score'].quantile(0.99)
# Panel_H1['net_sent_score_wins'] = Panel_H1['net_sent_score'].clip(lower=lower, upper=upper)

# lower = Panel_H1['Net_sent_score_weighted'].quantile(0.01)
# upper = Panel_H1['Net_sent_score_weighted'].quantile(0.99)
# Panel_H1['net_sent_score_weighted_wins'] = Panel_H1['Net_sent_score_weighted'].clip(lower=lower, upper=upper)




# Panel_H1 = Panel_H1[(Panel_H1['portfolio_var'] >= Panel_H1['portfolio_var'].quantile(0.01)) &
#                     (Panel_H1['portfolio_var'] <= Panel_H1['portfolio_var'].quantile(0.99))]


# lower = Panel_H1['portfolio_var'].quantile(0.01)
# upper = Panel_H1['portfolio_var'].quantile(0.99)
# Panel_H1['portfolio_var_wins'] = Panel_H1['portfolio_var'].clip(lower=lower, upper=upper)




Panel_H1 = Panel_H1[(Panel_H1['portfolio_vol'] >= Panel_H1['portfolio_vol'].quantile(0.01)) &
                    (Panel_H1['portfolio_vol'] <= Panel_H1['portfolio_vol'].quantile(0.99))]


lower = Panel_H1['portfolio_vol'].quantile(0.01)
upper = Panel_H1['portfolio_vol'].quantile(0.99)
Panel_H1['portfolio_vol_wins'] = Panel_H1['portfolio_vol'].clip(lower=lower, upper=upper)

Panel_H1['log_portfolio_vol'] = np.log(Panel_H1['portfolio_vol'])


Panel_H1['log_Negative'] = np.where(Panel_H1['Negative'] > 0, np.log(Panel_H1['Negative']), 0)
Panel_H1['log_Neutral'] = np.where(Panel_H1['Neutral'] > 0, np.log(Panel_H1['Neutral']), 0)
Panel_H1['log_Positive'] = np.where(Panel_H1['Positive'] > 0, np.log(Panel_H1['Positive']), 0)

Panel_H1['log_Negative_weighted'] = np.where(Panel_H1['Negative_weighted'] > 0, np.log(Panel_H1['Negative_weighted']), 0)
Panel_H1['log_Neutral_weighted'] = np.where(Panel_H1['Neutral_weighted'] > 0, np.log(Panel_H1['Neutral_weighted']), 0)
Panel_H1['log_Positive_weighted'] = np.where(Panel_H1['Positive_weighted'] > 0, np.log(Panel_H1['Positive_weighted']), 0)

Panel_H1['fund_size'] = Panel_H1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')


Panel_H1 = Panel_H1[Panel_H1['portfolio_vol'] != 0]

Panel_H1['log_portfolio_var'] = Panel_H1['log_portfolio_vol']*2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

# Drop NaN values from both columns
net_sent_score = Panel_H1['net_sent_score'].dropna()
net_sent_score_weighted = Panel_H1['Net_sent_score_weighted'].dropna()

# Define common bins using the range from both variables
common_min = min(net_sent_score.min(), net_sent_score_weighted.min())
common_max = max(net_sent_score.max(), net_sent_score_weighted.max())
bins = np.linspace(common_min, common_max, 51)  # 50 bins

plt.figure(figsize=(10, 6))
plt.hist(net_sent_score, bins=bins, edgecolor='black', alpha=0.5, label="Net sentiment score", color='skyblue')
plt.hist(net_sent_score_weighted, bins=bins, edgecolor='black', alpha=0.5, label="Net weighted sentiment score", color='salmon')

# Fit a normal distribution to net_sent_score data
mu, sigma = net_sent_score.mean(), net_sent_score.std()
x = np.linspace(common_min, common_max, 100)
pdf = norm.pdf(x, mu, sigma)

# Scale the PDF to match the histogram frequency counts
bin_width = (common_max - common_min) / 50
scaled_pdf = pdf * bin_width * len(net_sent_score)

plt.plot(x, scaled_pdf, 'k-', linewidth=2, label=f'Normal PDF (μ={mu:.2f}, σ={sigma:.2f})')

plt.xlabel("Sentiment score")
plt.ylabel("Frequency")
# plt.title("Histogram of Net Sentiment Score and Net Sent Score Weighted with Normal PDF")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

portfolio_var = Panel_H1['portfolio_vol'].dropna()

# Calculate 1st and 99th percentiles
lower_bound = portfolio_var.quantile(0.01)
upper_bound = portfolio_var.quantile(0.99)

# Winsorize the data by clipping extreme values
winsorized_portfolio_var = portfolio_var.clip(lower=lower_bound, upper=upper_bound)

plt.figure(figsize=(10, 6))
plt.hist(winsorized_portfolio_var, bins=100, edgecolor='black', color='skyblue')
plt.xlabel("Portfolio Variance (Winsorized)")
plt.ylabel("Frequency")
plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.figure(figsize=(10, 6))
plt.hist(Panel_H1['log_portfolio_vol'], bins=200, edgecolor='black', color='skyblue')
plt.xlabel("Log(Portfolio Volatility)")
plt.ylabel("Frequency")
plt.title("Histogram of Log(Portfolio Volatility) (Top 10% outliers removed)")
plt.show()

In [ ]:
models = run_panel_models(
    df=Panel_H1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/1.vol.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['log_Negative_weighted', 'log_Neutral_weighted', 'log_Positive_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/1.vol.3.xlsx', index=False)

## Regression on weights

In [ ]:
Panel_H1_1 = Panel.groupby(by=['fund_key','firm_in_text', 'date'], as_index=False).agg({
    'Класс активов': 'first',
    'Сумма, руб.': 'sum',
    'Доля, %': 'sum',
    'Net_sent_score_weighted': 'mean',
    'Positive_weighted': 'sum',
    'Negative_weighted': 'sum',
    'Neutral_weighted': 'sum',
    'Negative': 'first',
    'Neutral': 'first',
    'Positive': 'first',
    'Flow':'first',
    'Imoex':'first',
    'imoex_ret':'first',
    'Ruonia':'first',
    'USD_RUB':'first',
    'unique_funds':'first',
    'dummy_mar2014':'first',
    'dummy_feb2022':'first'
})

Panel_H1_1 = Panel_H1_1[(Panel_H1_1['Доля, %'] >= Panel_H1_1['Доля, %'].quantile(0.01)) &
                    (Panel_H1_1['Доля, %'] <= Panel_H1_1['Доля, %'].quantile(0.99))]

lower = Panel_H1_1['Доля, %'].quantile(0.01)
upper = Panel_H1_1['Доля, %'].quantile(0.99)
Panel_H1_1['Доля, %_wins'] = Panel_H1_1['Доля, %'].clip(lower=lower, upper=upper)

Panel_H1_1.where(Panel_H1_1['Доля, %_wins'] >0, inplace=True)

Panel_H1_1 = Panel_H1_1[(Panel_H1_1['Доля, %_wins'] >= Panel_H1_1['Доля, %_wins'].quantile(0.01)) &
                    (Panel_H1_1['Доля, %_wins'] <= Panel_H1_1['Доля, %_wins'].quantile(0.98))]

Panel_H1_1['net_sent_score'] = (Panel_H1_1['Positive'] - Panel_H1_1['Negative'])/(Panel_H1_1['Positive'] + Panel_H1_1['Negative'] + Panel_H1_1['Neutral'])

Panel_H1_1['log_Negative'] = np.where(Panel_H1_1['Negative'] > 0, np.log(Panel_H1_1['Negative']), 0)
Panel_H1_1['log_Neutral'] = np.where(Panel_H1_1['Neutral'] > 0, np.log(Panel_H1_1['Neutral']), 0)
Panel_H1_1['log_Positive'] = np.where(Panel_H1_1['Positive'] > 0, np.log(Panel_H1_1['Positive']), 0)

Panel_H1_1['log_Negative_weighted'] = np.where(Panel_H1_1['Negative_weighted'] > 0, np.log(Panel_H1_1['Negative_weighted']), 0)
Panel_H1_1['log_Neutral_weighted'] = np.where(Panel_H1_1['Neutral_weighted'] > 0, np.log(Panel_H1_1['Neutral_weighted']), 0)
Panel_H1_1['log_Positive_weighted'] = np.where(Panel_H1_1['Positive_weighted'] > 0, np.log(Panel_H1_1['Positive_weighted']), 0)

Panel_H1_1['fund_size'] = Panel_H1_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(Panel_H1_1['Доля, %_wins'].dropna(), bins=50, edgecolor='black', color='skyblue')
plt.xlabel("Доля")
plt.ylabel("Frequency")
plt.title("Histogram of Доля")
plt.show()

In [ ]:
models = run_panel_models(
    df=Panel_H1_1,
    entity_col='fund_key',
    y_var='Доля, %_wins',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/1.weights.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H1_1,
    entity_col='fund_key',
    y_var='Доля, %_wins',
    main_vars=['log_Negative_weighted', 'log_Positive_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/1.weights.3.xlsx', index=False)

## H2

In [ ]:
fund_assets_wide = pd.read_excel(r'../../Data/processed_all.xlsx')
fund_assets_wide['fund_key'] = fund_assets_wide['Название фонда'] + '__' + fund_assets_wide['Номер регистрации'].astype(str)
fund_assets_wide['СЧА на дату отчетности, руб.'] = pd.to_numeric(fund_assets_wide['СЧА на дату отчетности, руб.'], errors='coerce')
fund_assets_wide['date'] = pd.to_datetime(fund_assets_wide['date'])
fund_classes = fund_assets_wide[['Название фонда', 'Название УК', 'Номер регистрации', 'Тип фонда', 'Категория фонда']].drop_duplicates(subset='Номер регистрации')
fund_classes['fund_key'] = fund_classes['Название фонда'] + '__' + fund_classes['Номер регистрации'].astype(str)

In [ ]:
Panel_H2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H2['net_sent_score'] = (Panel_H2['Positive'] - Panel_H2['Negative'])/(Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
Panel_H2['Neg_share'] = Panel_H2['Negative'] / (Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
Panel_H2['Pos_share'] = Panel_H2['Positive'] / (Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
Panel_H2['RVI'].fillna(0, inplace=True)

Panel_H2['Neg_share_weighted'] = Panel_H2['Negative_weighted'] / (Panel_H2['Positive_weighted'] + Panel_H2['Negative_weighted'] + Panel_H2['Neutral_weighted'])
Panel_H2['Pos_share_weighted'] = Panel_H2['Positive_weighted'] / (Panel_H2['Positive_weighted'] + Panel_H2['Negative_weighted'] + Panel_H2['Neutral_weighted'])


Panel_H2 = pd.merge(Panel_H2, fund_classes[['Тип фонда', 'Категория фонда', 'fund_key']], on=['fund_key'], how='left')
Panel_H2 = Panel_H2[Panel_H2['Категория фонда'].isin(['Акции', 'Смешанный'])]


Panel_H2 = Panel_H2[(Panel_H2['fund_liquidity'] >= Panel_H2['fund_liquidity'].quantile(0.01)) &
                    (Panel_H2['fund_liquidity'] <= Panel_H2['fund_liquidity'].quantile(0.99))]


lower = Panel_H2['fund_liquidity'].quantile(0.01)
upper = Panel_H2['fund_liquidity'].quantile(0.99)
Panel_H2['fund_liquidity_wins'] = Panel_H2['fund_liquidity'].clip(lower=lower, upper=upper)

In [ ]:
Panel_H2['log_Negative'] = np.where(Panel_H2['Negative'] > 0, np.log(Panel_H2['Negative']), 0)
Panel_H2['log_Neutral'] = np.where(Panel_H2['Neutral'] > 0, np.log(Panel_H2['Neutral']), 0)
Panel_H2['log_Positive'] = np.where(Panel_H2['Positive'] > 0, np.log(Panel_H2['Positive']), 0)

Panel_H2['log_Negative_weighted'] = np.where(Panel_H2['Negative_weighted'] > 0, np.log(Panel_H2['Negative_weighted']), 0)
Panel_H2['log_Neutral_weighted'] = np.where(Panel_H2['Neutral_weighted'] > 0, np.log(Panel_H2['Neutral_weighted']), 0)
Panel_H2['log_Positive_weighted'] = np.where(Panel_H2['Positive_weighted'] > 0, np.log(Panel_H2['Positive_weighted']), 0)

Panel_H2['log_fund_liquidity'] = np.where(Panel_H2['fund_liquidity'] > 0, np.log(Panel_H2['fund_liquidity']), 0)

Panel_H2 = Panel_H2[Panel_H2['log_fund_liquidity'] != 0]
Panel_H2['fund_size'] = Panel_H2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract non-null and positive liquidity values before taking log
liquidity = Panel_H2['fund_liquidity'].dropna()
liquidity = liquidity[liquidity > 0]
log_liquidity = np.log(liquidity)

plt.figure(figsize=(10, 6))
plt.hist(log_liquidity, bins=50, edgecolor='black', color='skyblue')
plt.xlabel("log(Fund Liquidity)")
plt.ylabel("Frequency")
plt.title("Histogram of Log-Transformed Fund Liquidity")
plt.show()

In [ ]:
models = run_panel_models(
    df=Panel_H2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/2.liq.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['log_Negative_weighted', 'log_Neutral_weighted', 'log_Positive_weighted'],
     controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/2.liq.3.xlsx', index=False)

## H3

In [ ]:
fund_assets_wide = pd.read_excel(r'../../Data/processed_all.xlsx')
fund_assets_wide['fund_key'] = fund_assets_wide['Название фонда'] + '__' + fund_assets_wide['Номер регистрации'].astype(str)

In [ ]:
fund_assets_wide['Категория фонда'].unique()

In [ ]:
Panel_H3_1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H3_1['net_sent_score'] = (Panel_H3_1['Positive'] - Panel_H3_1['Negative'])/(Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])
# Panel_H3_1['is_stocks_fund'] = Panel_H3_1['fund_key'].apply(lambda fk: int(any(name in fk for name in stocks_funds)))
Panel_H3_1=pd.merge(Panel_H3_1, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H3_1['Категория фонда'] = Panel_H3_1['Категория фонда'].apply(lambda x: str(x).lower())

Panel_H3_1['is_stocks_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('акции' in x))
Panel_H3_1['is_bonds_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('облигации' in x))
Panel_H3_1['is_comb_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('смешанный' in x))

Panel_H3_1['Neg_share'] = Panel_H3_1['Negative'] / (Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])
Panel_H3_1['Pos_share'] = Panel_H3_1['Positive'] / (Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])

Panel_H3_1['Neg_share_weighted'] = Panel_H3_1['Negative_weighted'] / (Panel_H3_1['Positive_weighted'] + Panel_H3_1['Negative_weighted'] + Panel_H3_1['Neutral_weighted'])
Panel_H3_1['Pos_share_weighted'] = Panel_H3_1['Positive_weighted'] / (Panel_H3_1['Positive_weighted'] + Panel_H3_1['Negative_weighted'] + Panel_H3_1['Neutral_weighted'])

Panel_H3_1['portfolio_vol'] = np.sqrt(Panel_H3_1['portfolio_var'])

Panel_H3_1 = Panel_H3_1[(Panel_H3_1['portfolio_vol'] >= Panel_H3_1['portfolio_vol'].quantile(0.01)) &
                    (Panel_H3_1['portfolio_vol'] <= Panel_H3_1['portfolio_vol'].quantile(0.99))]

lower = Panel_H3_1['portfolio_vol'].quantile(0.01)
upper = Panel_H3_1['portfolio_vol'].quantile(0.99)
Panel_H3_1['portfolio_vol_wins'] = Panel_H3_1['portfolio_vol'].clip(lower=lower, upper=upper)




Panel_H3_1['log_Negative'] = np.where(Panel_H3_1['Negative'] > 0, np.log(Panel_H3_1['Negative']), 0)
Panel_H3_1['log_Neutral'] = np.where(Panel_H3_1['Neutral'] > 0, np.log(Panel_H3_1['Neutral']), 0)
Panel_H3_1['log_Positive'] = np.where(Panel_H3_1['Positive'] > 0, np.log(Panel_H3_1['Positive']), 0)

Panel_H3_1['log_Negative_weighted'] = np.where(Panel_H3_1['Negative_weighted'] > 0, np.log(Panel_H3_1['Negative_weighted']), 0)
Panel_H3_1['log_Neutral_weighted'] = np.where(Panel_H3_1['Neutral_weighted'] > 0, np.log(Panel_H3_1['Neutral_weighted']), 0)
Panel_H3_1['log_Positive_weighted'] = np.where(Panel_H3_1['Positive_weighted'] > 0, np.log(Panel_H3_1['Positive_weighted']), 0)

Panel_H3_1['log_portfolio_vol'] = np.where(Panel_H3_1['portfolio_vol'] > 0, np.log(Panel_H3_1['portfolio_vol']), 0)
# Panel_H3_1 = Panel_H3_1[Panel_H3_1['portfolio_vol'] != 0]

Panel_H3_1 = Panel_H3_1[Panel_H3_1['log_portfolio_vol'] != 0]
Panel_H3_1['fund_size'] = Panel_H3_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

Panel_H3_1['equity_x_netsent']    = Panel_H3_1['is_stocks_fund'] * Panel_H3_1['Net_sent_score_weighted']

In [ ]:
models = run_panel_models(
    df=Panel_H3_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted','equity_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/3.vol.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H3_1[Panel_H3_1['is_stocks_fund']==1],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H3_1[Panel_H3_1['is_stocks_fund']==0],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_two_coefs(
    df,
    entity_col,
    y_var,
    main_var,
    controls,
    group_var,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap differences in BOTH the current and lagged coefficients
    of `main_var` between group==1 and group==0.
    Returns a dict with arrays and summaries for current, lag, and their diffs.
    """
    df1 = df[df[group_var] == 1].copy()
    df0 = df[df[group_var] == 0].copy()
    keys1, keys0 = df1[entity_col].unique(), df0[entity_col].unique()

    def _draw(_):
        # resample funds
        draw1 = np.random.choice(keys1, size=len(keys1), replace=True)
        draw0 = np.random.choice(keys0, size=len(keys0), replace=True)
        samp1 = df1[df1[entity_col].isin(draw1)]
        samp0 = df0[df0[entity_col].isin(draw0)]

        # fit 'both' spec on each
        m1 = run_panel_models(
            samp1, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        m0 = run_panel_models(
            samp0, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']

        # extract current and lagged coeffs
        c1 = m1.params[main_var]
        c1L = m1.params[f"{main_var}_L{lag}"]
        c0 = m0.params[main_var]
        c0L = m0.params[f"{main_var}_L{lag}"]
        return float(c1), float(c1L), float(c0), float(c0L)

    # run bootstrap
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split into arrays
    eq_current = draws[:,0]
    eq_lag     = draws[:,1]
    ne_current = draws[:,2]
    ne_lag     = draws[:,3]

    # differences
    diff_current = eq_current - ne_current
    diff_lag     = eq_lag     - ne_lag

    # build summary
    def summarize(arr):
        return {
            'mean':   arr.mean(),
            'CI':     np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'equity_current': summarize(eq_current),
        'non_current':    summarize(ne_current),
        'diff_current':   summarize(diff_current),
        'equity_lag':     summarize(eq_lag),
        'non_lag':        summarize(ne_lag),
        'diff_lag':       summarize(diff_lag),
    }

# Usage example:
res = bootstrap_two_coefs(
    df=Panel_H3_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_var='Net_sent_score_weighted',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    group_var='is_stocks_fund',
    lag=1,
    n_boot=1000
)

# Print results
print("Current coef — Equity:",    res['equity_current'])
print("Current coef — Non-equity:",res['non_current'])
print("Δcurrent:",                 res['diff_current'])

print("Lagged coef — Equity:",     res['equity_lag'])
print("Lagged coef — Non-equity:", res['non_lag'])
print("Δlagged:",                  res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=20, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['Equity', 'Non-equity', 'Difference']
    curr = [
        (res['equity_current']['mean'], res['equity_current']['CI']),
        (res['non_current']['mean'],  res['non_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['equity_lag']['mean'],  res['equity_lag']['CI']),
        (res['non_lag']['mean'],     res['non_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min(lows), max([2,2])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
Panel_H3_2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H3_2['net_sent_score'] = (Panel_H3_2['Positive'] - Panel_H3_2['Negative'])/(Panel_H3_2['Positive'] + Panel_H3_2['Negative'] + Panel_H3_2['Neutral'])
# Panel_H3_2['is_stocks_fund'] = Panel_H3_2['fund_key'].apply(lambda fk: int(any(name in fk for name in stocks_funds)))
Panel_H3_2=pd.merge(Panel_H3_2, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H3_2['Категория фонда'] = Panel_H3_2['Категория фонда'].apply(lambda x: str(x).lower())

Panel_H3_2['is_stocks_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('акции' in x))
Panel_H3_2['is_bonds_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('облигации' in x))
Panel_H3_2['is_comb_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('смешанный' in x))

Panel_H3_2['Neg_share_weighted'] = Panel_H3_2['Negative_weighted'] / (Panel_H3_2['Positive_weighted'] + Panel_H3_2['Negative_weighted'] + Panel_H3_2['Neutral_weighted'])
Panel_H3_2['Pos_share_weighted'] = Panel_H3_2['Positive_weighted'] / (Panel_H3_2['Positive_weighted'] + Panel_H3_2['Negative_weighted'] + Panel_H3_2['Neutral_weighted'])


Panel_H3_2 = Panel_H3_2[(Panel_H3_2['fund_liquidity'] >= Panel_H3_2['fund_liquidity'].quantile(0.01)) &
                    (Panel_H3_2['fund_liquidity'] <= Panel_H3_2['fund_liquidity'].quantile(0.99))]


lower = Panel_H3_2['fund_liquidity'].quantile(0.01)
upper = Panel_H3_2['fund_liquidity'].quantile(0.99)
Panel_H3_2['fund_liquidity_wins'] = Panel_H3_2['fund_liquidity'].clip(lower=lower, upper=upper)



Panel_H3_2['log_Negative'] = np.where(Panel_H3_2['Negative'] > 0, np.log(Panel_H3_2['Negative']), 0)
Panel_H3_2['log_Neutral'] = np.where(Panel_H3_2['Neutral'] > 0, np.log(Panel_H3_2['Neutral']), 0)
Panel_H3_2['log_Positive'] = np.where(Panel_H3_2['Positive'] > 0, np.log(Panel_H3_2['Positive']), 0)

Panel_H3_2['log_Negative_weighted'] = np.where(Panel_H3_2['Negative_weighted'] > 0, np.log(Panel_H3_2['Negative_weighted']), 0)
Panel_H3_2['log_Neutral_weighted'] = np.where(Panel_H3_2['Neutral_weighted'] > 0, np.log(Panel_H3_2['Neutral_weighted']), 0)
Panel_H3_2['log_Positive_weighted'] = np.where(Panel_H3_2['Positive_weighted'] > 0, np.log(Panel_H3_2['Positive_weighted']), 0)

Panel_H3_2['log_fund_liquidity'] = np.where(Panel_H3_2['fund_liquidity'] > 0, np.log(Panel_H3_2['fund_liquidity']), 0)
Panel_H3_2 = Panel_H3_2[Panel_H3_2['fund_liquidity'] != 0]

Panel_H3_2 = Panel_H3_2[Panel_H3_2['log_fund_liquidity'] != 0]
Panel_H3_2['fund_size'] = Panel_H3_2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

Panel_H3_2['equity_x_netsent']    = Panel_H3_2['is_stocks_fund'] * Panel_H3_2['Net_sent_score_weighted']

In [ ]:
models = run_panel_models(
    df=Panel_H3_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted','equity_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/3.liq.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H3_2[Panel_H3_2['is_stocks_fund']==1],
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H3_2[Panel_H3_2['is_stocks_fund']==0],
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_two_coefs(
    df,
    entity_col,
    y_var,
    main_var,
    controls,
    group_var,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap differences in BOTH the current and lagged coefficients
    of `main_var` between group==1 and group==0.
    Returns a dict with arrays and summaries for current, lag, and their diffs.
    """
    df1 = df[df[group_var] == 1].copy()
    df0 = df[df[group_var] == 0].copy()
    keys1, keys0 = df1[entity_col].unique(), df0[entity_col].unique()

    def _draw(_):
        # resample funds
        draw1 = np.random.choice(keys1, size=len(keys1), replace=True)
        draw0 = np.random.choice(keys0, size=len(keys0), replace=True)
        samp1 = df1[df1[entity_col].isin(draw1)]
        samp0 = df0[df0[entity_col].isin(draw0)]

        # fit 'both' spec on each
        m1 = run_panel_models(
            samp1, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        m0 = run_panel_models(
            samp0, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']

        # extract current and lagged coeffs
        c1 = m1.params[main_var]
        c1L = m1.params[f"{main_var}_L{lag}"]
        c0 = m0.params[main_var]
        c0L = m0.params[f"{main_var}_L{lag}"]
        return float(c1), float(c1L), float(c0), float(c0L)

    # run bootstrap
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split into arrays
    eq_current = draws[:,0]
    eq_lag     = draws[:,1]
    ne_current = draws[:,2]
    ne_lag     = draws[:,3]

    # differences
    diff_current = eq_current - ne_current
    diff_lag     = eq_lag     - ne_lag

    # build summary
    def summarize(arr):
        return {
            'mean':   arr.mean(),
            'CI':     np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'equity_current': summarize(eq_current),
        'non_current':    summarize(ne_current),
        'diff_current':   summarize(diff_current),
        'equity_lag':     summarize(eq_lag),
        'non_lag':        summarize(ne_lag),
        'diff_lag':       summarize(diff_lag),
    }

# Usage example:
res = bootstrap_two_coefs(
    df=Panel_H3_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_var='Net_sent_score_weighted',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    group_var='is_stocks_fund',
    lag=1,
    n_boot=1000
)

# Print results
print("Current coef — Equity:",    res['equity_current'])
print("Current coef — Non-equity:",res['non_current'])
print("Δcurrent:",                 res['diff_current'])

print("Lagged coef — Equity:",     res['equity_lag'])
print("Lagged coef — Non-equity:", res['non_lag'])
print("Δlagged:",                  res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['Equity', 'Non-equity', 'Difference']
    curr = [
        (res['equity_current']['mean'], res['equity_current']['CI']),
        (res['non_current']['mean'],  res['non_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['equity_lag']['mean'],  res['equity_lag']['CI']),
        (res['non_lag']['mean'],     res['non_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-3,-3]), max([4,4])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


## H4

In [ ]:
Panel_H4_1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H4_1['net_sent_score'] = (Panel_H4_1['Positive'] - Panel_H4_1['Negative'])/(Panel_H4_1['Positive'] + Panel_H4_1['Negative'] + Panel_H4_1['Positive'])
Panel_H4_1=pd.merge(Panel_H4_1, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H4_1['Категория фонда'] = Panel_H4_1['Категория фонда'].apply(lambda x: str(x).lower())

fund_assets_wide['СЧА на дату отчетности, руб.'] = pd.to_numeric(fund_assets_wide['СЧА на дату отчетности, руб.'], errors='coerce')
fund_assets_wide['Итого сумма активов, руб.'] = pd.to_numeric(fund_assets_wide['Итого сумма активов, тыс. руб.'], errors='coerce') * 1000
fund_assets_wide['date'] = pd.to_datetime(fund_assets_wide['date'])

fund_assets_wide['size'] = fund_assets_wide[['СЧА на дату отчетности, руб.', 'Итого сумма активов, руб.']].max(axis=1)

q = 0.9

fund_assets_wide['is_small'] = 0
fund_assets_wide['is_small'] = fund_assets_wide.groupby('date')['size'].transform(lambda x: (x <= x.quantile(q)).astype(int))
Panel_H4_1=pd.merge(Panel_H4_1, fund_assets_wide.groupby(['fund_key', 'date']).agg({'is_small':'first'}), on = ['fund_key', 'date'], how = 'left')
Panel_H4_1['is_small'].fillna(0, inplace=True)

Panel_H4_1['fund_size'] = Panel_H4_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')
Panel_H4_1['is_small'] = Panel_H4_1['fund_size'].transform(lambda x: (x <= x.quantile(q)).astype(int))



Panel_H4_1 = Panel_H4_1[(Panel_H4_1['portfolio_var'] >= Panel_H4_1['portfolio_var'].quantile(0.01)) &
                    (Panel_H4_1['portfolio_var'] <= Panel_H4_1['portfolio_var'].quantile(0.99))]

Panel_H4_1['portfolio_vol'] = np.sqrt(Panel_H4_1['portfolio_var'])

Panel_H4_1 = Panel_H4_1[(Panel_H4_1['portfolio_vol'] >= Panel_H4_1['portfolio_vol'].quantile(0.01)) &
                    (Panel_H4_1['portfolio_vol'] <= Panel_H4_1['portfolio_vol'].quantile(0.99))]

lower = Panel_H4_1['portfolio_vol'].quantile(0.01)
upper = Panel_H4_1['portfolio_vol'].quantile(0.99)
Panel_H4_1['portfolio_vol_wins'] = Panel_H4_1['portfolio_vol'].clip(lower=lower, upper=upper)


Panel_H4_1['log_Negative'] = np.where(Panel_H4_1['Negative'] > 0, np.log(Panel_H4_1['Negative']), 0)
Panel_H4_1['log_Neutral'] = np.where(Panel_H4_1['Neutral'] > 0, np.log(Panel_H4_1['Neutral']), 0)
Panel_H4_1['log_Positive'] = np.where(Panel_H4_1['Positive'] > 0, np.log(Panel_H4_1['Positive']), 0)

Panel_H4_1['log_Negative_weighted'] = np.where(Panel_H4_1['Negative_weighted'] > 0, np.log(Panel_H4_1['Negative_weighted']), 0)
Panel_H4_1['log_Neutral_weighted'] = np.where(Panel_H4_1['Neutral_weighted'] > 0, np.log(Panel_H4_1['Neutral_weighted']), 0)
Panel_H4_1['log_Positive_weighted'] = np.where(Panel_H4_1['Positive_weighted'] > 0, np.log(Panel_H4_1['Positive_weighted']), 0)

Panel_H4_1['log_portfolio_vol'] = np.where(Panel_H4_1['portfolio_vol'] > 0, np.log(Panel_H4_1['portfolio_vol']), 0)
# Panel_H4_1 = Panel_H4_1[Panel_H4_1['portfolio_vol'] != 0]

Panel_H4_1 = Panel_H4_1[Panel_H4_1['log_portfolio_vol'] != 0]
Panel_H4_1['fund_size'] = Panel_H4_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

Panel_H4_1['small_x_netsent']    = Panel_H4_1['is_small'] * Panel_H4_1['Net_sent_score_weighted']
Panel_H4_1['fund_size'] = Panel_H4_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')


In [ ]:
models = run_panel_models(
    df=Panel_H4_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted','small_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/4.vol.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H4_1[Panel_H4_1['is_small']==1],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H4_1[Panel_H4_1['is_small']==0],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_two_coefs(
    df,
    entity_col,
    y_var,
    main_var,
    controls,
    group_var,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap differences in BOTH the current and lagged coefficients
    of `main_var` between group==1 and group==0.
    Returns a dict with arrays and summaries for current, lag, and their diffs.
    """
    df1 = df[df[group_var] == 1].copy()
    df0 = df[df[group_var] == 0].copy()
    keys1, keys0 = df1[entity_col].unique(), df0[entity_col].unique()

    def _draw(_):
        # resample funds
        draw1 = np.random.choice(keys1, size=len(keys1), replace=True)
        draw0 = np.random.choice(keys0, size=len(keys0), replace=True)
        samp1 = df1[df1[entity_col].isin(draw1)]
        samp0 = df0[df0[entity_col].isin(draw0)]

        # fit 'both' spec on each
        m1 = run_panel_models(
            samp1, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        m0 = run_panel_models(
            samp0, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']

        # extract current and lagged coeffs
        c1 = m1.params[main_var]
        c1L = m1.params[f"{main_var}_L{lag}"]
        c0 = m0.params[main_var]
        c0L = m0.params[f"{main_var}_L{lag}"]
        return float(c1), float(c1L), float(c0), float(c0L)

    # run bootstrap
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split into arrays
    eq_current = draws[:,0]
    eq_lag     = draws[:,1]
    ne_current = draws[:,2]
    ne_lag     = draws[:,3]

    # differences
    diff_current = eq_current - ne_current
    diff_lag     = eq_lag     - ne_lag

    # build summary
    def summarize(arr):
        return {
            'mean':   arr.mean(),
            'CI':     np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'small_current': summarize(eq_current),
        'non_current':    summarize(ne_current),
        'diff_current':   summarize(diff_current),
        'small_lag':     summarize(eq_lag),
        'non_lag':        summarize(ne_lag),
        'diff_lag':       summarize(diff_lag),
    }

# Usage example:
res = bootstrap_two_coefs(
    df=Panel_H4_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_var='Net_sent_score_weighted',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    group_var='is_small',
    lag=1,
    n_boot=1000
)

# Print results
print("Current coef — small:",    res['small_current'])
print("Current coef — Non-small:",res['non_current'])
print("Δcurrent:",                 res['diff_current'])

print("Lagged coef — small:",     res['small_lag'])
print("Lagged coef — Non-small:", res['non_lag'])
print("Δlagged:",                  res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['small', 'Non-small', 'Difference']
    curr = [
        (res['small_current']['mean'], res['small_current']['CI']),
        (res['non_current']['mean'],  res['non_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['small_lag']['mean'],  res['small_lag']['CI']),
        (res['non_lag']['mean'],     res['non_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-2,-2]), max([2,2])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
Panel_H4_2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H4_2['net_sent_score'] = (Panel_H4_2['Positive'] - Panel_H4_2['Negative'])/(Panel_H4_2['Positive'] + Panel_H4_2['Negative'] + Panel_H4_2['Neutral'])
Panel_H4_2=pd.merge(Panel_H4_2, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H4_2['Категория фонда'] = Panel_H4_2['Категория фонда'].apply(lambda x: str(x).lower())

fund_assets_wide['СЧА на дату отчетности, руб.'] = pd.to_numeric(fund_assets_wide['СЧА на дату отчетности, руб.'], errors='coerce')
fund_assets_wide['Итого сумма активов, руб.'] = pd.to_numeric(fund_assets_wide['Итого сумма активов, тыс. руб.'], errors='coerce') * 1000
fund_assets_wide['date'] = pd.to_datetime(fund_assets_wide['date'])

fund_assets_wide['is_small'] = fund_assets_wide.groupby('date')['size'].transform(lambda x: (x <= x.quantile(0.8)).astype(int))
Panel_H4_2=pd.merge(Panel_H4_2, fund_assets_wide.groupby(['fund_key', 'date']).agg({'is_small':'first'}), on = ['fund_key', 'date'], how = 'left')
Panel_H4_2['is_small'].fillna(1, inplace=True)

q = 0.8
Panel_H4_2['fund_size'] = Panel_H4_2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')
Panel_H4_2['is_small'] = Panel_H4_2['fund_size'].transform(lambda x: (x <= x.quantile(q)).astype(int))

Panel_H4_2 = Panel_H4_2[(Panel_H4_2['fund_liquidity'] >= Panel_H4_2['fund_liquidity'].quantile(0.01)) &
                    (Panel_H4_2['fund_liquidity'] <= Panel_H4_2['fund_liquidity'].quantile(0.99))]


lower = Panel_H4_2['fund_liquidity'].quantile(0.01)
upper = Panel_H4_2['fund_liquidity'].quantile(0.99)
Panel_H4_2['fund_liquidity_wins'] = Panel_H4_2['fund_liquidity'].clip(lower=lower, upper=upper)


Panel_H4_2['log_Negative'] = np.where(Panel_H4_2['Negative'] > 0, np.log(Panel_H4_2['Negative']), 0)
Panel_H4_2['log_Neutral'] = np.where(Panel_H4_2['Neutral'] > 0, np.log(Panel_H4_2['Neutral']), 0)
Panel_H4_2['log_Positive'] = np.where(Panel_H4_2['Positive'] > 0, np.log(Panel_H4_2['Positive']), 0)

Panel_H4_2['log_Negative_weighted'] = np.where(Panel_H4_2['Negative_weighted'] > 0, np.log(Panel_H4_2['Negative_weighted']), 0)
Panel_H4_2['log_Neutral_weighted'] = np.where(Panel_H4_2['Neutral_weighted'] > 0, np.log(Panel_H4_2['Neutral_weighted']), 0)
Panel_H4_2['log_Positive_weighted'] = np.where(Panel_H4_2['Positive_weighted'] > 0, np.log(Panel_H4_2['Positive_weighted']), 0)

Panel_H4_2['log_fund_liquidity'] = np.where(Panel_H4_2['fund_liquidity'] > 0, np.log(Panel_H4_2['fund_liquidity']), 0)
Panel_H4_2 = Panel_H4_2[Panel_H4_2['log_fund_liquidity'] != 0]
Panel_H4_2['small_x_netsent']    = Panel_H4_2['is_small'] * Panel_H4_2['Net_sent_score_weighted']


Panel_H4_2['fund_size'] = Panel_H4_2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')


In [ ]:
models = run_panel_models(
    df=Panel_H4_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted','small_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/4.liq.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H4_1[Panel_H4_1['is_small']==1],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H4_1[Panel_H4_1['is_small']==0],
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_two_coefs(
    df,
    entity_col,
    y_var,
    main_var,
    controls,
    group_var,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap differences in BOTH the current and lagged coefficients
    of `main_var` between group==1 and group==0.
    Returns a dict with arrays and summaries for current, lag, and their diffs.
    """
    df1 = df[df[group_var] == 1].copy()
    df0 = df[df[group_var] == 0].copy()
    keys1, keys0 = df1[entity_col].unique(), df0[entity_col].unique()

    def _draw(_):
        # resample funds
        draw1 = np.random.choice(keys1, size=len(keys1), replace=True)
        draw0 = np.random.choice(keys0, size=len(keys0), replace=True)
        samp1 = df1[df1[entity_col].isin(draw1)]
        samp0 = df0[df0[entity_col].isin(draw0)]

        # fit 'both' spec on each
        m1 = run_panel_models(
            samp1, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        m0 = run_panel_models(
            samp0, entity_col, y_var, [main_var], controls,
            lag=lag, entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']

        # extract current and lagged coeffs
        c1 = m1.params[main_var]
        c1L = m1.params[f"{main_var}_L{lag}"]
        c0 = m0.params[main_var]
        c0L = m0.params[f"{main_var}_L{lag}"]
        return float(c1), float(c1L), float(c0), float(c0L)

    # run bootstrap
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split into arrays
    eq_current = draws[:,0]
    eq_lag     = draws[:,1]
    ne_current = draws[:,2]
    ne_lag     = draws[:,3]

    # differences
    diff_current = eq_current - ne_current
    diff_lag     = eq_lag     - ne_lag

    # build summary
    def summarize(arr):
        return {
            'mean':   arr.mean(),
            'CI':     np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'small_current': summarize(eq_current),
        'non_current':    summarize(ne_current),
        'diff_current':   summarize(diff_current),
        'small_lag':     summarize(eq_lag),
        'non_lag':        summarize(ne_lag),
        'diff_lag':       summarize(diff_lag),
    }

# Usage example:
res = bootstrap_two_coefs(
    df=Panel_H4_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_var='Net_sent_score_weighted',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    group_var='is_small',
    lag=1,
    n_boot=1000
)

# Print results
print("Current coef — small:",    res['small_current'])
print("Current coef — Non-small:",res['non_current'])
print("Δcurrent:",                 res['diff_current'])

print("Lagged coef — small:",     res['small_lag'])
print("Lagged coef — Non-small:", res['non_lag'])
print("Δlagged:",                  res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['small', 'Non-small', 'Difference']
    curr = [
        (res['small_current']['mean'], res['small_current']['CI']),
        (res['non_current']['mean'],  res['non_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['small_lag']['mean'],  res['small_lag']['CI']),
        (res['non_lag']['mean'],     res['non_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-7,-7]), max([8,8])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


## H5

In [ ]:
Panel_H5 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ftc, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H5['net_sent_score'] = (Panel_H5['Positive'] - Panel_H5['Negative'])/(Panel_H5['Positive'] + Panel_H5['Negative'] + Panel_H5['Neutral'])
Panel_H5['net_sent_score_reporting'] = (Panel_H5['earnings_Positive'] - Panel_H5['earnings_Negative'])/(Panel_H5['earnings_Positive'] + Panel_H5['earnings_Negative'] + Panel_H5['earnings_Neutral'])
Panel_H5['net_sent_score_forecasts'] = (Panel_H5['forecast_Positive'] - Panel_H5['forecast_Negative'])/(Panel_H5['forecast_Positive'] + Panel_H5['forecast_Negative'] + Panel_H5['forecast_Neutral'])
Panel_H5['net_sent_score_sanctions'] = (Panel_H5['sanctions_Positive'] - Panel_H5['sanctions_Negative'])/(Panel_H5['sanctions_Positive'] + Panel_H5['sanctions_Negative'] + Panel_H5['sanctions_Neutral'])
Panel_H5['net_sent_score_others'] = (Panel_H5['other_Positive'] - Panel_H5['other_Negative'])/(Panel_H5['other_Positive'] + Panel_H5['other_Negative'] + Panel_H5['other_Neutral'])

Panel_H5['portfolio_vol'] = np.sqrt(Panel_H5['portfolio_var'])

Panel_H5 = Panel_H5[(Panel_H5['portfolio_vol'] >= Panel_H5['portfolio_vol'].quantile(0.01)) &
                    (Panel_H5['portfolio_vol'] <= Panel_H5['portfolio_vol'].quantile(0.99))]


lower = Panel_H5['portfolio_vol'].quantile(0.01)
upper = Panel_H5['portfolio_vol'].quantile(0.99)
Panel_H5['portfolio_vol_wins'] = Panel_H5['portfolio_vol'].clip(lower=lower, upper=upper)


Panel_H5['log_Negative'] = np.where(Panel_H5['Negative'] > 0, np.log(Panel_H5['Negative']), 0)
Panel_H5['log_Neutral'] = np.where(Panel_H5['Neutral'] > 0, np.log(Panel_H5['Neutral']), 0)
Panel_H5['log_Positive'] = np.where(Panel_H5['Positive'] > 0, np.log(Panel_H5['Positive']), 0)

Panel_H5['log_Negative_weighted'] = np.where(Panel_H5['Negative_weighted'] > 0, np.log(Panel_H5['Negative_weighted']), 0)
Panel_H5['log_Neutral_weighted'] = np.where(Panel_H5['Neutral_weighted'] > 0, np.log(Panel_H5['Neutral_weighted']), 0)
Panel_H5['log_Positive_weighted'] = np.where(Panel_H5['Positive_weighted'] > 0, np.log(Panel_H5['Positive_weighted']), 0)

Panel_H5['log_portfolio_vol'] = np.where(Panel_H5['portfolio_vol'] > 0, np.log(Panel_H5['portfolio_vol']), 0)

Panel_H5 = Panel_H5[Panel_H5['log_portfolio_vol'] != 0]
Panel_H5['fund_size'] = Panel_H5.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')




In [ ]:
models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/5.vol.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['net_sent_score_reporting'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_reporting'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_reporting',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (reporting) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Financial reports', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-1,-1]), max([1.5,1.5])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['net_sent_score_forecasts'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_forecasts'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_forecasts',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (forecasts) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Forecasts', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-2,-2]), max([3,3])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['net_sent_score_sanctions'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_forecasts'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=False, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_sanctions',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (sanctions) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Sanctions', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-0.5,-0.5]), max([1.5,1.5])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
Panel_H5_2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ftc, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H5_2['net_sent_score'] = (Panel_H5_2['Positive'] - Panel_H5_2['Negative'])/(Panel_H5_2['Positive'] + Panel_H5_2['Negative'] + Panel_H5_2['Neutral'])
Panel_H5_2['net_sent_score_reporting'] = (Panel_H5_2['earnings_Positive'] - Panel_H5_2['earnings_Negative'])/(Panel_H5_2['earnings_Positive'] + Panel_H5_2['earnings_Negative'] + Panel_H5_2['earnings_Neutral'])
Panel_H5_2['net_sent_score_forecasts'] = (Panel_H5_2['forecast_Positive'] - Panel_H5_2['forecast_Negative'])/(Panel_H5_2['forecast_Positive'] + Panel_H5_2['forecast_Negative'] + Panel_H5_2['forecast_Neutral'])
Panel_H5_2['net_sent_score_sanctions'] = (Panel_H5_2['sanctions_Positive'] - Panel_H5_2['sanctions_Negative'])/(Panel_H5_2['sanctions_Positive'] + Panel_H5_2['sanctions_Negative'] + Panel_H5_2['sanctions_Neutral'])
Panel_H5_2['net_sent_score_others'] = (Panel_H5_2['other_Positive'] - Panel_H5_2['other_Negative'])/(Panel_H5_2['other_Positive'] + Panel_H5_2['other_Negative'] + Panel_H5_2['other_Neutral'])

Panel_H5_2 = Panel_H5_2[(Panel_H5_2['fund_liquidity'] >= Panel_H5_2['fund_liquidity'].quantile(0.01)) &
                    (Panel_H5_2['fund_liquidity'] <= Panel_H5_2['fund_liquidity'].quantile(0.99))]


lower = Panel_H5_2['fund_liquidity'].quantile(0.01)
upper = Panel_H5_2['fund_liquidity'].quantile(0.99)
Panel_H5_2['fund_liquidity_wins'] = Panel_H5_2['fund_liquidity'].clip(lower=lower, upper=upper)


Panel_H5_2['log_Negative'] = np.where(Panel_H5_2['Negative'] > 0, np.log(Panel_H5_2['Negative']), 0)
Panel_H5_2['log_Neutral'] = np.where(Panel_H5_2['Neutral'] > 0, np.log(Panel_H5_2['Neutral']), 0)
Panel_H5_2['log_Positive'] = np.where(Panel_H5_2['Positive'] > 0, np.log(Panel_H5_2['Positive']), 0)

Panel_H5_2['log_Negative_weighted'] = np.where(Panel_H5_2['Negative_weighted'] > 0, np.log(Panel_H5_2['Negative_weighted']), 0)
Panel_H5_2['log_Neutral_weighted'] = np.where(Panel_H5_2['Neutral_weighted'] > 0, np.log(Panel_H5_2['Neutral_weighted']), 0)
Panel_H5_2['log_Positive_weighted'] = np.where(Panel_H5_2['Positive_weighted'] > 0, np.log(Panel_H5_2['Positive_weighted']), 0)

Panel_H5_2['log_fund_liquidity'] = np.where(Panel_H5_2['fund_liquidity'] > 0, np.log(Panel_H5_2['fund_liquidity']), 0)

Panel_H5_2 = Panel_H5_2[Panel_H5_2['log_fund_liquidity'] != 0]
Panel_H5_2['fund_size'] = Panel_H5_2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')




In [ ]:
models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/5.liq.1.xlsx', index=False)

In [ ]:
models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['net_sent_score_reporting'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_reporting'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_reporting',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (reporting) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Financial reports', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-1,-1]), max([3,3])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['net_sent_score_forecasts'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_reporting'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_forecasts',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (forecasts) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Forecasts', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-2,-2]), max([3,3])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


In [ ]:
models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

models = run_panel_models(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['net_sent_score_sanctions'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=True,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models['both'])

In [ ]:
import numpy as np
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def bootstrap_model_diff(
    df,
    entity_col,
    y_var,
    varA,        # e.g. 'Net_sent_score_weighted'
    varB,        # e.g. 'net_sent_score_reporting'
    controls,
    lag=1,
    n_boot=500,
    n_jobs=-1
):
    """
    Bootstrap the difference in coefficients between two 'both' models:
      - Model A has main_var = varA
      - Model B has main_var = varB
    Returns summaries for current and lagged coefficient differences.
    """
    # keep unique entity list
    entities = df[entity_col].unique()

    def _one_draw(_):
        # resample entities with replacement
        draw = np.random.choice(entities, size=len(entities), replace=True)
        samp = df[df[entity_col].isin(draw)]
        # fit both-spec for A
        resA = run_panel_models(
            samp, entity_col, y_var, [varA], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # fit both-spec for B
        resB = run_panel_models(
            samp, entity_col, y_var, [varB], controls,
            lag=lag,
            entity_effects=True, time_effects=False,
            cov_type='clustered', cluster_entity=True, cluster_time=True
        )['both']
        # extract
        cA  = float(resA.params[varA])
        cA1 = float(resA.params[f"{varA}_L{lag}"])
        cB  = float(resB.params[varB])
        cB1 = float(resB.params[f"{varB}_L{lag}"])
        return cA, cA1, cB, cB1

    # run bootstrap in parallel with progress bar
    draws = np.array(Parallel(n_jobs=n_jobs)(
        delayed(_one_draw)(i) for i in tqdm(range(n_boot), desc="Bootstrapping")
    ))
    # split arrays
    A_curr, A_lag, B_curr, B_lag = draws.T
    d_curr = A_curr - B_curr
    d_lag  = A_lag  - B_lag

    # helper to summarize
    def summarize(arr):
        return {
            'mean':  arr.mean(),
            'CI':    np.percentile(arr, [2.5, 97.5]),
            'p_val': 2 * min((arr>=0).mean(), (arr<=0).mean())
        }

    return {
        'A_current': summarize(A_curr),
        'B_current': summarize(B_curr),
        'diff_current': summarize(d_curr),
        'A_lag': summarize(A_lag),
        'B_lag': summarize(B_lag),
        'diff_lag': summarize(d_lag),
    }

# — Usage example —
res = bootstrap_model_diff(
    df=Panel_H5_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    varA='Net_sent_score_weighted',
    varB='net_sent_score_sanctions',
    controls=[
      'Flow','Imoex','imoex_ret',
      'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    n_boot=1000
)

# Print summaries:
print("Model A (weighted) current coef:",    res['A_current'])
print("Model B (sanctions) current coef:",   res['B_current'])
print("Difference current Δβ:",              res['diff_current'])
print("Model A lagged coef:",                res['A_lag'])
print("Model B lagged coef:",                res['B_lag'])
print("Difference lagged Δβ:",               res['diff_lag'])


In [ ]:
import matplotlib.pyplot as plt

def plot_bootstrap_cis(res, figsize=(14, 5), fontsize=28, pad_ratio=0.1):
    """
    Plot current and lagged coefficient CIs side by side, 
    annotating both lower and upper CI bounds.
    
    res: dict from bootstrap_two_coefs()
    figsize: tuple for figure size
    fontsize: int for font size of labels and annotations
    pad_ratio: fraction of CI range to pad the y-axis by
    """
    # temporarily increase font size
    plt.rcParams.update({'font.size': fontsize})

    labels = ['All news', 'Sanctions', 'Difference']
    curr = [
        (res['A_current']['mean'], res['A_current']['CI']),
        (res['B_current']['mean'],  res['B_current']['CI']),
        (res['diff_current']['mean'], res['diff_current']['CI']),
    ]
    lagg = [
        (res['A_lag']['mean'],  res['A_lag']['CI']),
        (res['B_lag']['mean'],     res['B_lag']['CI']),
        (res['diff_lag']['mean'],    res['diff_lag']['CI']),
    ]

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)

    def draw(ax, data, title):
        x = range(len(data))
        lows, highs = [], []
        for i, (mean, ci) in enumerate(data):
            lo, hi = ci
            err_low, err_high = mean - lo, hi - mean
            lows.append(lo)
            highs.append(hi)
            ax.errorbar(
                x[i], mean,
                yerr=[[err_low], [err_high]],
                fmt='o-', capsize=5, markersize=8
            )
            # annotate lower and upper bounds without clipping
            ax.annotate(f"{lo:.3f}", xy=(x[i], lo),
                        xytext=(0, -10), textcoords='offset points',
                        ha='center', va='top', clip_on=False)
            ax.annotate(f"{hi:.3f}", xy=(x[i], hi),
                        xytext=(0, 10), textcoords='offset points',
                        ha='center', va='bottom', clip_on=False)

        # set y-limits with padding
        y_min, y_max = min([-1.5,-1.5]), max([3.5,3.5])
        pad = (y_max - y_min) * pad_ratio
        ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title)
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Group')
        ax.grid(True, axis='y', linestyle=':', alpha=0.7)

    draw(axes[0], curr, 'Current Coefficient 95% CI')
    draw(axes[1], lagg, 'Lagged Coefficient 95% CI')

    axes[0].set_ylabel('Coefficient estimate')
    plt.tight_layout()
    plt.show()

    # restore defaults
    plt.rcdefaults()

# Usage
plot_bootstrap_cis(res, figsize=(24,10), fontsize=20)


# Preliminary evidence

## H0

In [ ]:
fund_assets_wide = pd.read_excel(r'../../Data/processed_all.xlsx')
fund_assets_wide['fund_key'] = fund_assets_wide['Название фонда'] + '__' + fund_assets_wide['Номер регистрации'].astype(str)
fund_assets_wide['СЧА на дату отчетности, руб.'] = pd.to_numeric(fund_assets_wide['СЧА на дату отчетности, руб.'], errors='coerce')
fund_assets_wide['date'] = pd.to_datetime(fund_assets_wide['date'])
fund_classes = fund_assets_wide[['Название фонда', 'Название УК', 'Номер регистрации', 'Тип фонда', 'Категория фонда']].drop_duplicates(subset='Номер регистрации')
fund_classes['fund_key'] = fund_classes['Название фонда'] + '__' + fund_classes['Номер регистрации'].astype(str)

In [ ]:
pif_prices = pd.read_csv(r'../../Data/Y/funds_01012015_17052025.csv', sep = ';', encoding='cp1251',low_memory=False).dropna(subset=['Фонд'])
pif_prices['Дата'] = pd.to_datetime(pif_prices['Дата'])
pif_prices.sort_values(['Фонд', 'Дата'], ascending=[False, True], inplace=True)
pif_prices['fund_ret'] = pif_prices.groupby('Фонд')['Пай'].pct_change()

Panel_etf = pd.merge(Panel, pif_prices[['Дата', 'Фонд', 'Пай', 'СЧА']], left_on=['date', 'Название фонда'], right_on=['Дата', 'Фонд'], how='left')
Panel_etf.sort_values(['Номер регистрации', 'date'],ascending=[False, True], inplace=True)
Panel_etf = pd.merge(Panel_etf, fund_classes[['Тип фонда', 'Категория фонда', 'fund_key']], on=['fund_key'], how='left')

Panel_etf['net_sent_score'] = (Panel_etf['Positive'] - Panel_etf['Negative'])/(Panel_etf['Positive'] + Panel_etf['Negative'] + Panel_etf['Neutral'])

In [ ]:
agg = Panel_etf.groupby(['fund_key', 'date'], as_index=False)['Пай'].mean()
agg['fund_ret'] = agg.groupby('fund_key')['Пай'].pct_change()
Panel_etf = Panel_etf.merge(agg[['fund_key', 'date', 'fund_ret']], on=['fund_key', 'date'], how='left')

In [ ]:
Panel_etf['excess_return'] = np.where(
    Panel_etf['Категория фонда'].isin(['Акции', 'Смешанный']),
    Panel_etf['fund_ret'] - Panel_etf['imoex_ret'],
    None
)

In [ ]:
Panel_agg_ftc = Panel_etf.groupby(['fund_key', 'date'], as_index=False).agg({
    'Класс активов': 'sum',
    'Сумма, руб.': 'sum',
    'Доля, %': 'sum',
    'Net_sent_score_weighted': 'sum',
    'Positive_weighted': 'sum',
    'Negative_weighted': 'sum',
    'Neutral_weighted': 'sum',
    'Negative': 'sum',
    'Neutral':  'sum',
    'Positive': 'sum',
    'forecast_Negative': 'sum',
    'forecast_Neutral':  'sum',
    'forecast_Positive': 'sum',
    'earnings_Negative':'sum',
    'earnings_Neutral': 'sum',
    'earnings_Positive':'sum',
    'sanctions_Negative':'sum',
    'sanctions_Neutral':'sum',
    'sanctions_Positive':'sum',
    'other_Negative':   'sum',
    'other_Neutral':    'sum',
    'other_Positive':   'sum',
    'Flow': 'first',
    'Imoex': 'first',
    'imoex_log_ret': 'first',
    'imoex_ret': 'first',
    'Ruonia': 'first',
    'USD_RUB': 'first',
    'unique_funds': 'first',
    'dummy_mar2014': 'first',
    'dummy_feb2022': 'first',
    'fund_ret' : 'first',
    'excess_return' : 'first'
})

Panel_agg_ftc['net_sent_score'] = (Panel_agg_ftc['Positive'] - Panel_agg_ftc['Negative'])/(Panel_agg_ftc['Positive'] + Panel_agg_ftc['Negative'] + Panel_agg_ftc['Neutral'])

Panel_agg_ftc['fund_size'] = Panel_agg_ftc.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

In [ ]:
Panel_H0 = Panel_agg_ftc

# Panel_H0 = Panel_H0[(Panel_H0['net_sent_score'] >= Panel_H0['net_sent_score'].quantile(0.01)) &
#                     (Panel_H0['net_sent_score'] <= Panel_H0['net_sent_score'].quantile(0.99))]

# Panel_H0 = Panel_H0[(Panel_H0['Net_sent_score_weighted'] >= Panel_H0['Net_sent_score_weighted'].quantile(0.01)) &
#                     (Panel_H0['Net_sent_score_weighted'] <= Panel_H0['Net_sent_score_weighted'].quantile(0.99))]

Panel_H0 = Panel_H0[(Panel_H0['excess_return'] >= Panel_H0['excess_return'].quantile(0.05)) &
                    (Panel_H0['excess_return'] <= Panel_H0['excess_return'].quantile(0.95))]

Panel_H0 = Panel_H0[(Panel_H0['fund_ret'] >= Panel_H0['fund_ret'].quantile(0.05)) &
                    (Panel_H0['fund_ret'] <= Panel_H0['fund_ret'].quantile(0.95))]

# lower = Panel_H0['net_sent_score'].quantile(0.01)
# upper = Panel_H0['net_sent_score'].quantile(0.99)
# Panel_H0['net_sent_score_wins'] = Panel_H0['net_sent_score'].clip(lower=lower, upper=upper)

# lower = Panel_H0['Net_sent_score_weighted'].quantile(0.01)
# upper = Panel_H0['Net_sent_score_weighted'].quantile(0.99)
# Panel_H0['net_sent_score_weighted_wins'] = Panel_H0['Net_sent_score_weighted'].clip(lower=lower, upper=upper)

lower = Panel_H0['excess_return'].quantile(0.01)
upper = Panel_H0['excess_return'].quantile(0.99)
Panel_H0['excess_return_wins'] = Panel_H0['excess_return'].clip(lower=lower, upper=upper)


lower = Panel_H0['fund_ret'].quantile(0.01)
upper = Panel_H0['fund_ret'].quantile(0.99)
Panel_H0['fund_ret_wins'] = Panel_H0['fund_ret'].clip(lower=lower, upper=upper)

Panel_H0['log_excess_return'] = Panel_H0['excess_return_wins'].apply(lambda x: 0 if x == 0 else np.log(x + 1))
Panel_H0['log_fund_ret'] = Panel_H0['fund_ret_wins'].apply(lambda x: 0 if x == 0 else np.log(x + 1))

# Panel_H0 = Panel_H0[Panel_H0['fund_ret_wins'] != 0]

In [ ]:
import matplotlib.pyplot as plt

portfolio_var = Panel_H0['excess_return'].dropna()

# Calculate 1st and 99th percentiles
lower_bound = portfolio_var.quantile(0.01)
upper_bound = portfolio_var.quantile(0.99)

# Winsorize the data by clipping extreme values
winsorized_portfolio_var = portfolio_var.clip(lower=lower_bound, upper=upper_bound)

plt.figure(figsize=(10, 6))
plt.hist(winsorized_portfolio_var, bins=50, edgecolor='black', color='skyblue')
# plt.xlabel("Portfolio Variance (Winsorized)")
# plt.ylabel("Frequency")
# plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
# model for net_sent_score
models = run_panel_models(
    df=Panel_H0,
    entity_col='fund_key',
    y_var='log_excess_return',
    main_vars=['net_sent_score'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/0.excess.s.xlsx', index=False)

In [ ]:
# model for Net_sent_score_weighted
models = run_panel_models(
    df=Panel_H0,
    entity_col='fund_key',
    y_var='log_excess_return',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/0.excess.w.xlsx', index=False)

In [ ]:
# model for net_sent_score
models = run_panel_models(
    df=Panel_H0,
    entity_col='fund_key',
    y_var='log_fund_ret',
    main_vars=['net_sent_score'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/0.simple.s.xlsx', index=False)

In [ ]:
# model for net_sent_score
models = run_panel_models(
    df=Panel_H0,
    entity_col='fund_key',
    y_var='log_fund_ret',
    main_vars=['Net_sent_score_weighted'],
    controls=[
        'Flow','Imoex', 'imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_feb2022'
    ],
    lag=1
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/0.simple.w.xlsx', index=False)

In [ ]:
Panel_H0_sr = pd.merge(Panel_H0, Panel_H1[['portfolio_var_wins','date', 'fund_key']], on=['date', 'fund_key'], how='left')
Panel_H0_sr['sharp_ratio'] = Panel_H0_sr['excess_return_wins'] / Panel_H0_sr['portfolio_var_wins']
Panel_H0_sr = Panel_H0_sr[(Panel_H0_sr['sharp_ratio'] >= Panel_H0_sr['sharp_ratio'].quantile(0.1)) &
                    (Panel_H0_sr['sharp_ratio'] <= Panel_H0_sr['sharp_ratio'].quantile(0.9))]

In [ ]:
import matplotlib.pyplot as plt

portfolio_var = Panel_H0_sr['sharp_ratio'].dropna()

# Calculate 1st and 99th percentiles
lower_bound = portfolio_var.quantile(0.01)
upper_bound = portfolio_var.quantile(0.99)

# Winsorize the data by clipping extreme values
winsorized_portfolio_var = portfolio_var.clip(lower=lower_bound, upper=upper_bound)

plt.figure(figsize=(10, 6))
plt.hist(winsorized_portfolio_var, bins=50, edgecolor='black', color='skyblue')
# plt.xlabel("Portfolio Variance (Winsorized)")
# plt.ylabel("Frequency")
# plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

df = Panel_H0_sr.copy()
df['date'] = pd.to_datetime(df['date'])
df = df.set_index(['fund_key', 'date'])

i = 1
df = df.sort_values(['fund_key', 'date']).assign(**{f'net_sent_score_L{i}': lambda d, lag=i: d.groupby('fund_key')['net_sent_score'].shift(lag)})

Y = df['sharp_ratio']
X = ['net_sent_score']
L_X = [
    # f'news_count_lag{i-1}',
    f'net_sent_score_L{i}']
# L_X = []

control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB', 'unique_funds', 'dummy_feb2022'] 

exog = sm.add_constant(df[X + L_X + control_vars])

model = PanelOLS(Y, exog, entity_effects=True, time_effects=False, drop_absorbed=True)
results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

print(results)


# ================================
# Hausman specification test
# ================================

from linearmodels.panel import RandomEffects
import numpy as np
import scipy.stats as st

# 1. Re-use the prepared exogenous matrix (exog) and dependent variable (Y)

# 2. Estimate the random-effects (RE) model with the same regressors
re_mod   = RandomEffects(Y, exog)
re_res   = re_mod.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# 3. Collect coefficient vectors and covariance matrices
b_fe  = results.params
b_re  = re_res.params

V_fe  = results.cov
V_re  = re_res.cov

# 4. Compute the Hausman statistic:  (b_FE – b_RE)' [V_FE – V_RE]^{-1} (b_FE – b_RE)
#    — keep only the common coefficients (drop any that were absorbed or not estimated in either model)

common = b_fe.index.intersection(b_re.index)
dbeta  = b_fe[common] - b_re[common]
Vdiff  = V_fe.loc[common, common] - V_re.loc[common, common]

stat   = float(dbeta.T @ np.linalg.inv(Vdiff) @ dbeta)
df_h   = len(common)                          # degrees of freedom
pvalue = 1 - st.chi2.cdf(stat, df_h)

print("Hausman test  χ²({}) = {:.3f}".format(df_h, stat))
print("p-value           = {:.4f}".format(pvalue))

if pvalue < 0.05:
    print("→ Reject H0: Random-effects inconsistent; prefer fixed-effects.")
else:
    print("→ Fail to reject H0: Random-effects consistent; RE may be used.")

print(re_res)

results_df = results_to_df_two_nonzero(re_res)
results_df.to_excel(r"../../Code/Analisys/regr_res/0.1.xlsx", index=True, engine="openpyxl")


# Plots

In [ ]:
Panel_H1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H1['net_sent_score'] = (Panel_H1['Positive'] - Panel_H1['Negative'])/(Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Neg_share'] = Panel_H1['Negative'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Pos_share'] = Panel_H1['Positive'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])

Panel_H1['Neg_share_weighted'] = Panel_H1['Negative_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])
Panel_H1['Pos_share_weighted'] = Panel_H1['Positive_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])

Panel_H1['log_portfolio_var'] = np.log(Panel_H1['portfolio_var'])
Panel_H1.loc[Panel_H1['portfolio_var'] == 0, 'log_portfolio_var'] = 0
Panel_H1 = Panel_H1[Panel_H1['portfolio_var'] > 0]
Panel_H1['portfolio_vol'] = np.sqrt(Panel_H1['portfolio_var'])


In [ ]:
import matplotlib.pyplot as plt

portfolio_var = Panel_H1['portfolio_var'].dropna()

# Calculate 1st and 99th percentiles
lower_bound = portfolio_var.quantile(0.01)
upper_bound = portfolio_var.quantile(0.99)

# Winsorize the data by clipping extreme values
winsorized_portfolio_var = portfolio_var.clip(lower=lower_bound, upper=upper_bound)

plt.figure(figsize=(10, 6))
plt.hist(portfolio_var, bins=100, edgecolor='black', color='skyblue')
plt.xlabel("Portfolio Variance")
plt.ylabel("Frequency")
# plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
Panel_H1 = Panel_H1[(Panel_H1['portfolio_var'] >= Panel_H1['portfolio_var'].quantile(0.01)) &
                    (Panel_H1['portfolio_var'] <= Panel_H1['portfolio_var'].quantile(0.99))]

In [ ]:
import matplotlib.pyplot as plt

portfolio_var = Panel_H1['portfolio_var'].dropna()

plt.figure(figsize=(10, 6))
plt.hist(portfolio_var, bins=100, edgecolor='black', color='skyblue')
plt.xlabel("Portfolio Variance")
plt.ylabel("Frequency")
# plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

portfolio_var = np.log(Panel_H1['portfolio_var'].dropna())

plt.figure(figsize=(10, 6))
plt.hist(portfolio_var, bins=100, edgecolor='black', color='skyblue')
plt.xlabel("Portfolio Variance")
plt.ylabel("Frequency")
# plt.title("Histogram of Winsorized Portfolio Variance")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.size'] = 20  # Change font size


Panel_H1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])

Panel_H1['portfolio_vol'] = np.sqrt(Panel_H1['portfolio_var'])

Panel_H1['net_sent_score'] = (Panel_H1['Positive'] - Panel_H1['Negative'])/(Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Neg_share'] = Panel_H1['Negative'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])
Panel_H1['Pos_share'] = Panel_H1['Positive'] / (Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral'])

Panel_H1['Neg_share_weighted'] = Panel_H1['Negative_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])
Panel_H1['Pos_share_weighted'] = Panel_H1['Positive_weighted'] / (Panel_H1['Positive_weighted'] + Panel_H1['Negative_weighted'] + Panel_H1['Neutral_weighted'])

Panel_H1['log_portfolio_vol'] = np.log(Panel_H1['portfolio_vol'])
Panel_H1.loc[Panel_H1['portfolio_vol'] == 0, 'log_portfolio_vol'] = 0
Panel_H1 = Panel_H1[Panel_H1['portfolio_vol'] > 0]

# Make a copy of Panel_H1 before outlier removal (this is the "initial" data)
Panel_H1_initial = Panel_H1.copy()

# Now apply outlier removal (1st and 99th percentiles) on Portfolio Volatility in Panel_H1
Panel_H1_filtered = Panel_H1[
    (Panel_H1['portfolio_vol'] >= Panel_H1['portfolio_vol'].quantile(0.01)) &
    (Panel_H1['portfolio_vol'] <= Panel_H1['portfolio_vol'].quantile(0.99))
]

# Prepare series for plotting
raw_portfolio_vol = Panel_H1_initial['portfolio_vol'].dropna()
filtered_portfolio_vol = Panel_H1_filtered['portfolio_vol'].dropna()
# Log-transform raw portfolio Volatility (only where > 0)
log_portfolio_vol = np.log(raw_portfolio_vol[raw_portfolio_vol > 0])

# Create three subplots in one row
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

# Plot (1): Raw Portfolio Volatility histogram (initial data)
axs[0].hist(raw_portfolio_vol, bins=100, edgecolor='black', color='skyblue')
axs[0].set_xlabel("Portfolio Volatility")
axs[0].set_ylabel("Frequency")
axs[0].set_title("Portfolio Volatility")

# Plot (2): Portfolio Volatility histogram after outlier removal
axs[1].hist(filtered_portfolio_vol, bins=100, edgecolor='black', color='skyblue')
axs[1].set_xlabel("Portfolio Volatility")
# axs[1].set_ylabel("Frequency")
axs[1].set_title("Portfolio Volatility with outliers removed")

# Plot (3): Log-transformed Portfolio Volatility histogram (using raw data)
axs[2].hist(log_portfolio_vol, bins=100, edgecolor='black', color='skyblue')
axs[2].set_xlabel("Log(Portfolio Volatility)")
# axs[2].set_ylabel("Frequency")
axs[2].set_title("Log of Portfolio Volatility")

plt.tight_layout()
plt.show()
plt.rcdefaults()  # Revert font settings to default

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# fund_assets_wide = pd.read_excel(r'../../Data/processed_all.xlsx')
# fund_assets_wide['fund_key'] = fund_assets_wide['Название фонда'] + '__' + fund_assets_wide['Номер регистрации'].astype(str)
# fund_assets_wide['СЧА на дату отчетности, руб.'] = pd.to_numeric(fund_assets_wide['СЧА на дату отчетности, руб.'], errors='coerce')
# fund_assets_wide['date'] = pd.to_datetime(fund_assets_wide['date'])
# fund_classes = fund_assets_wide[['Название фонда', 'Название УК', 'Номер регистрации', 'Тип фонда', 'Категория фонда']].drop_duplicates(subset='Номер регистрации')
# fund_classes['fund_key'] = fund_classes['Название фонда'] + '__' + fund_classes['Номер регистрации'].astype(str)
# Panel_H2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
# Panel_H2['net_sent_score'] = (Panel_H2['Positive'] - Panel_H2['Negative'])/(Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
# Panel_H2['Neg_share'] = Panel_H2['Negative'] / (Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
# Panel_H2['Pos_share'] = Panel_H2['Positive'] / (Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral'])
# Panel_H2['RVI'].fillna(0, inplace=True)

# Panel_H2 = pd.merge(Panel_H2, fund_classes[['Тип фонда', 'Категория фонда', 'fund_key']], on=['fund_key'], how='left')
# Panel_H2 = Panel_H2[Panel_H2['Категория фонда'].isin(['Акции', 'Смешанный'])]

plt.rcParams['font.size'] = 20  # Change font size



# Assume Panel_H2 is already defined and filtered appropriately (only for 'Акции' and 'Смешанный')

# Extract the raw fund liquidity values (only positive values for log transformation)
raw_fund_liquidity = Panel_H2['fund_liquidity'].dropna()
raw_fund_liquidity = raw_fund_liquidity[raw_fund_liquidity > 0]

# Remove outliers by filtering out values below the 1st percentile or above the 99th percentile
lower_bound = raw_fund_liquidity.quantile(0.01)
upper_bound = raw_fund_liquidity.quantile(0.99)
filtered_fund_liquidity = raw_fund_liquidity[(raw_fund_liquidity >= lower_bound) & (raw_fund_liquidity <= upper_bound)]

# Log-transform the raw fund liquidity values
log_fund_liquidity = np.log(raw_fund_liquidity)

# Create three subplots in one row
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

# (1) Raw Fund Liquidity histogram
axs[0].hist(raw_fund_liquidity, bins=100, edgecolor='black', color='skyblue')
axs[0].set_xlabel("Fund Liquidity")
axs[0].set_ylabel("Frequency")
axs[0].set_title("Fund Liquidity")

# (2) Filtered Fund Liquidity histogram (outliers removed)
axs[1].hist(filtered_fund_liquidity, bins=100, edgecolor='black', color='skyblue')
axs[1].set_xlabel("Fund Liquidity")
axs[1].set_title("Fund Liquidity with Outliers Removed")

# (3) Log-transformed Fund Liquidity histogram
axs[2].hist(log_fund_liquidity, bins=100, edgecolor='black', color='skyblue')
axs[2].set_xlabel("Log(Fund Liquidity)")
axs[2].set_title("Log of Fund Liquidity")

plt.tight_layout()
plt.show()

plt.rcdefaults()  # Revert font settings to default

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams['font.size'] = 20  # Change font size

# Assume Panel_agg_ftc is the original DataFrame containing the 'excess_return' variable.
# Extract raw excess return data (unfiltered)
raw_excess_return = Panel_agg_ftc['excess_return'].dropna()

# Filter the data by removing outliers using the 1st and 99th percentiles
lower_bound = raw_excess_return.quantile(0.01)
upper_bound = raw_excess_return.quantile(0.99)
filtered_excess_return = raw_excess_return[(raw_excess_return >= lower_bound) & (raw_excess_return <= upper_bound)]

# Compute a log-transformation using log(1 + excess_return)
# (This transformation is common for returns, and it handles negative values provided excess returns > -1)
log_excess_return = raw_excess_return.apply(lambda x: np.log(x + 1))
# Create three subplots in one row
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

# (1) Raw Excess Return histogram (unfiltered)
axs[0].hist(raw_excess_return, bins=100, edgecolor='black', color='skyblue')
axs[0].set_title("Excess Return")
axs[0].set_xlabel("Excess Return")
axs[0].set_ylabel("Frequency")

# (2) Filtered Excess Return histogram (outliers removed)
axs[1].hist(filtered_excess_return, bins=100, edgecolor='black', color='skyblue')
axs[1].set_title("Excess Return with Outliers Removed")
axs[1].set_xlabel("Excess Return")
# axs[1].set_ylabel("Frequency")

# (3) Log-transformed Excess Return histogram (using log(1+excess_return))
axs[2].hist(log_excess_return, bins=100, edgecolor='black', color='skyblue')
axs[2].set_title("Log of Excess Return")
axs[2].set_xlabel("Log(Excess Return)")
# axs[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()
plt.rcdefaults()  # Revert font settings to default

In [ ]:
(filtered_excess_return).astype(float)

In [ ]:
import pandas as pd

# Ensure the series are float type
filtered_excess_return = filtered_excess_return.astype(float)
log_excess_return = log_excess_return.astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": filtered_excess_return.quantile(0.05),
    "25%": filtered_excess_return.quantile(0.25),
    "Mean": filtered_excess_return.mean(),
    "Median": filtered_excess_return.median(),
    "75%": filtered_excess_return.quantile(0.75),
    "95%": filtered_excess_return.quantile(0.95),
    "Variance": filtered_excess_return.var(),
}

# Compute statistics for the log-transformed excess return
stats_log = {
    "5%": log_excess_return.quantile(0.05),
    "25%": log_excess_return.quantile(0.25),
    "Mean": log_excess_return.mean(),
    "Median": log_excess_return.median(),
    "75%": log_excess_return.quantile(0.75),
    "95%": log_excess_return.quantile(0.95),
    "Variance": log_excess_return.var(),
}

# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw, stats_log], 
                        index=['excess_return', 'log_excess_return']).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
filtered_portfolio_vol = filtered_portfolio_vol.astype(float)
log_portfolio_vol = log_portfolio_vol.astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": filtered_portfolio_vol.quantile(0.05),
    "25%": filtered_portfolio_vol.quantile(0.25),
    "Mean": filtered_portfolio_vol.mean(),
    "Median": filtered_portfolio_vol.median(),
    "75%": filtered_portfolio_vol.quantile(0.75),
    "95%": filtered_portfolio_vol.quantile(0.95),
    "Variance": filtered_portfolio_vol.var(),
}

# Compute statistics for the log-transformed excess return
stats_log = {
    "5%": log_portfolio_vol.quantile(0.05),
    "25%": log_portfolio_vol.quantile(0.25),
    "Mean": log_portfolio_vol.mean(),
    "Median": log_portfolio_vol.median(),
    "75%": log_portfolio_vol.quantile(0.75),
    "95%": log_portfolio_vol.quantile(0.95),
    "Variance": log_portfolio_vol.var(),
}

# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw, stats_log], 
                        index=['excess_return', 'log_portfolio_vol']).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
filtered_fund_liquidity = filtered_fund_liquidity.astype(float)*1000000000
log_fund_liquidity = log_fund_liquidity.astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": filtered_fund_liquidity.quantile(0.05),
    "25%": filtered_fund_liquidity.quantile(0.25),
    "Mean": filtered_fund_liquidity.mean(),
    "Median": filtered_fund_liquidity.median(),
    "75%": filtered_fund_liquidity.quantile(0.75),
    "95%": filtered_fund_liquidity.quantile(0.95),
    "Variance": filtered_fund_liquidity.var(),
}

# Compute statistics for the log-transformed excess return
stats_log = {
    "5%": log_fund_liquidity.quantile(0.05),
    "25%": log_fund_liquidity.quantile(0.25),
    "Mean": log_fund_liquidity.mean(),
    "Median": log_fund_liquidity.median(),
    "75%": log_fund_liquidity.quantile(0.75),
    "95%": log_fund_liquidity.quantile(0.95),
    "Variance": log_fund_liquidity.var(),
}

# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw, stats_log], 
                        index=['excess_return', 'log_fund_liquidity']).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
data = Panel_H1['net_sent_score'].astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": data.quantile(0.05),
    "25%": data.quantile(0.25),
    "Mean": data.mean(),
    "Median": data.median(),
    "75%": data.quantile(0.75),
    "95%": data.quantile(0.95),
    "Variance": data.var(),
}



# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw]).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
data = Panel_H1['Net_sent_score_weighted'].astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": data.quantile(0.05),
    "25%": data.quantile(0.25),
    "Mean": data.mean(),
    "Median": data.median(),
    "75%": data.quantile(0.75),
    "95%": data.quantile(0.95),
    "Variance": data.var(),
}



# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw]).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
data = Panel_H1['log_Positive_weighted'].astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": data.quantile(0.05),
    "25%": data.quantile(0.25),
    "Mean": data.mean(),
    "Median": data.median(),
    "75%": data.quantile(0.75),
    "95%": data.quantile(0.95),
    "Variance": data.var(),
}



# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw]).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
data = Panel_H1['log_Neutral_weighted'].astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": data.quantile(0.05),
    "25%": data.quantile(0.25),
    "Mean": data.mean(),
    "Median": data.median(),
    "75%": data.quantile(0.75),
    "95%": data.quantile(0.95),
    "Variance": data.var(),
}



# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw]).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Ensure the series are float type
data = Panel_H1['log_Negative_weighted'].astype(float)

# Compute statistics for the raw filtered excess return
stats_raw = {
    "5%": data.quantile(0.05),
    "25%": data.quantile(0.25),
    "Mean": data.mean(),
    "Median": data.median(),
    "75%": data.quantile(0.75),
    "95%": data.quantile(0.95),
    "Variance": data.var(),
}



# Create a single DataFrame with two rows: one for the raw and one for the log-transformed statistics
stats_df = pd.DataFrame([stats_raw]).round(3)
print(stats_df)

In [ ]:
import pandas as pd

# Assume 'controls' is the DataFrame containing our control variables
# and that it includes a 'date' column that we want to exclude
control_vars = ['Flow', 'Imoex','imoex_ret', 'Ruonia', 'USD_RUB', 'RVI', 'unique_funds']
stats_names = ["5%", "25%", "Mean", "Median", "75%", "95%", "Variance"]
controls1 = controls

controls1['Flow'] = controls['Flow']*1000
# Initialize dictionary to hold stats for each control variable
stats_dict = {}

for var in control_vars:
    series = controls1[var].dropna().astype(float)
    stats_dict[var] = {
        "5%": series.quantile(0.05),
        "25%": series.quantile(0.25),
        "Mean": series.mean(),
        "Median": series.median(),
        "75%": series.quantile(0.75),
        "95%": series.quantile(0.95),
        "Std": series.std()
    }

# Convert the stats dictionary to a DataFrame
stats_df = pd.DataFrame(stats_dict).T.round(4)
print(stats_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters for two risky assets ---
mu_s, mu_b = 0.2, 0.12        # Expected returns: stocks vs bonds
sigma_s, sigma_b = 0.25, 0.05  # Volatilities
rho = 0.2                      # Correlation
cov_sb = rho * sigma_s * sigma_b

rf = 0.1  # risk-free rate

# --- Generate the full hyperbolic frontier (allowing leverage/shorting) ---
w = np.linspace(-1, 2, 500)  # portfolio weight on stocks
mu_p = w*mu_s + (1 - w)*mu_b
sigma_p = np.sqrt(
    w**2 * sigma_s**2 +
    (1 - w)**2 * sigma_b**2 +
    2 * w * (1 - w) * cov_sb
)

# --- Find the tangency (max Sharpe) portfolio ---
sharpe = (mu_p - rf) / sigma_p
i_tan = np.nanargmax(sharpe)
w_t, sigma_t, mu_t = w[i_tan], sigma_p[i_tan], mu_p[i_tan]

# --- Capital Allocation Line (CAL) / CML ---
x_line = np.linspace(0, sigma_p.max(), 100)
cal_slope = (mu_t - rf) / sigma_t
y_line = rf + cal_slope * x_line

# --- Plotting ---
plt.figure(figsize=(8, 6))
# Hyperbolic frontier (both branches)
plt.plot(sigma_p, mu_p, color='navy', lw=2)

# Tangent line
plt.plot(x_line, y_line, color='gray', lw=2, ls='--')

# Mark optimal (tangency) portfolio
plt.scatter(sigma_t, mu_t, color='black', s=80, zorder=5)
plt.text(sigma_t + 0.005, mu_t + 0.005,
         'Optimal Portfolio', fontsize=12)

# Labels on axes
plt.xlabel('Volatility / Standard Deviation', fontsize=12)
plt.ylabel('Expected Return', fontsize=12)

# Annotate risk-free level
plt.axhline(rf, color='gray', ls=':', lw=1)
plt.text(0.01, rf + 0.002, '', fontsize=12)

# Annotate CAL and frontier
plt.text(0.18, rf + cal_slope*0.18 + 0.01,
         'Capital Allocation Line', color='gray', fontsize=8)
plt.text(0.4, mu_p.max() - 0.02,
         'Efficient Frontier', color='navy', fontsize=8)

# Style
plt.grid(linestyle=':', alpha=0.6)
plt.xlim(0, sigma_p.max() + 0.02)
plt.ylim(0, mu_p.max() + 0.05)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Base parameters for two assets ---
mu_s, mu_b       = 0.3, 0.21       # expected returns
σ_s, σ_b         = 0.25, 0.05       # volatilities
ρ                = 0.2
cov_sb           = ρ * σ_s * σ_b
rf               = 0.20             # risk-free rate

# weight grid
w         = np.linspace(-1, 2, 500)
# base frontier
μ_base    = w*mu_s + (1-w)*mu_b
σ_base    = np.sqrt(w**2*σ_s**2 + (1-w)**2*σ_b**2 + 2*w*(1-w)*cov_sb)

# shifts
Δμ, Δσ    = 0.01, 0.03

# shifted frontiers
μ_good    = μ_base + Δμ
σ_good    = σ_base + Δσ
μ_bad     = μ_base - Δμ
σ_bad     = σ_base - Δσ

def tangency(μ_p, σ_p):
    """Find tangency weight, σ, μ for frontier"""
    sharpe = (μ_p - rf)/σ_p
    i       = np.nanargmax(sharpe)
    return σ_p[i], μ_p[i]

# tangency points
σ0, μ0    = tangency(μ_base, σ_base)
σ1, μ1    = tangency(μ_good, σ_good)
σ2, μ2    = tangency(μ_bad,  σ_bad)

# CAL lines
x_cal     = np.linspace(0, max(σ_bad.max(), σ_good.max(), σ_base.max()), 200)
def cal(σ_t, μ_t):
    slope = (μ_t - rf)/σ_t
    return rf + slope*x_cal

y0        = cal(σ0, μ0)
y1        = cal(σ1, μ1)
y2        = cal(σ2, μ2)

# --- Plot ---
plt.figure(figsize=(8,6))

# frontiers
plt.plot(σ_base, μ_base, color='navy', label='Neutral', lw=2)
plt.plot(σ_good, μ_good, color='green', label='Positive News', lw=2)
plt.plot(σ_bad,  μ_bad,  color='red',   label='Negative News', lw=2)

# CALs
plt.plot(x_cal, y0, color='navy',  ls='--', lw=1.5)
plt.plot(x_cal, y1, color='green', ls='--', lw=1.5)
plt.plot(x_cal, y2, color='red',   ls='--', lw=1.5)

# tangency points
plt.scatter([σ0],[μ0], color='navy',  s=60, zorder=5)
plt.scatter([σ1],[μ1], color='green', s=60, zorder=5)
plt.scatter([σ2],[μ2], color='red',   s=60, zorder=5)

# annotations
plt.text(σ0-0.005, μ0-0.01, 'Opt (neutral)', color='navy')
plt.text(σ1+0.005, μ1-0.005, 'Opt (+ news)', color='green')
plt.text(σ2-0.02, μ2+0.005, 'Opt (– news)', color='red')

# risk-free line
plt.axhline(rf, color='gray', ls=':', lw=1)
plt.text(0.01, rf+0.003, '', fontsize=10)

# labels & legend
plt.xlabel('Volatility (σ)', fontsize=12)
plt.ylabel('Expected Return (μ)', fontsize=12)
# plt.title('Efficient Frontiers under Different News Sentiment', fontsize=14)
plt.legend()
plt.grid(linestyle=':', alpha=0.6)

# set limits so all three frontiers fit
maxσ = max(σ_base.max(), σ_good.max())
maxμ = max(μ_base.max(), μ_good.max())
plt.xlim(0, 0.2)
plt.ylim(0.1, 0.3)

plt.tight_layout()
plt.show()


# Other

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- re‐prepare just the two columns we care about ---
df_vif = Panel_H1.copy()
df_vif['date'] = pd.to_datetime(df_vif['date'])
df_vif = df_vif.set_index(['fund_key','date']).sort_index()

# make the lag exactly as in run_panel_models
df_vif['Net_sent_score_weighted_L1'] = (
    df_vif.groupby(level='fund_key')['Net_sent_score_weighted']
          .shift(1)
)
df_vif = df_vif.dropna(subset=['Net_sent_score_weighted_L1'])

# build the exog matrix for VIF
exog = df_vif[['Net_sent_score_weighted','Net_sent_score_weighted_L1']]
exog = sm.add_constant(exog)

# compute VIFs
vif_df = (
    pd.DataFrame({
        'variable': exog.columns,
        'VIF': [variance_inflation_factor(exog.values, i)
                for i in range(exog.shape[1])]
    })
)
print(vif_df)


In [ ]:
Panel_H3_1 = pd.merge(panel_df[['date', 'fund_key', 'portfolio_var']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H3_1['net_sent_score'] = (Panel_H3_1['Positive'] - Panel_H3_1['Negative'])/(Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])
# Panel_H3_1['is_stocks_fund'] = Panel_H3_1['fund_key'].apply(lambda fk: int(any(name in fk for name in stocks_funds)))
Panel_H3_1=pd.merge(Panel_H3_1, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H3_1['Категория фонда'] = Panel_H3_1['Категория фонда'].apply(lambda x: str(x).lower())

Panel_H3_1['is_stocks_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('акции' in x))
Panel_H3_1['is_bonds_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('облигации' in x))
Panel_H3_1['is_comb_fund'] = Panel_H3_1['Категория фонда'].apply(lambda x: int('смешанный' in x))

Panel_H3_1['Neg_share'] = Panel_H3_1['Negative'] / (Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])
Panel_H3_1['Pos_share'] = Panel_H3_1['Positive'] / (Panel_H3_1['Positive'] + Panel_H3_1['Negative'] + Panel_H3_1['Neutral'])

Panel_H3_1['Neg_share_weighted'] = Panel_H3_1['Negative_weighted'] / (Panel_H3_1['Positive_weighted'] + Panel_H3_1['Negative_weighted'] + Panel_H3_1['Neutral_weighted'])
Panel_H3_1['Pos_share_weighted'] = Panel_H3_1['Positive_weighted'] / (Panel_H3_1['Positive_weighted'] + Panel_H3_1['Negative_weighted'] + Panel_H3_1['Neutral_weighted'])

Panel_H3_1['portfolio_vol'] = np.sqrt(Panel_H3_1['portfolio_var'])

Panel_H3_1 = Panel_H3_1[(Panel_H3_1['portfolio_vol'] >= Panel_H3_1['portfolio_vol'].quantile(0.01)) &
                    (Panel_H3_1['portfolio_vol'] <= Panel_H3_1['portfolio_vol'].quantile(0.99))]

lower = Panel_H3_1['portfolio_vol'].quantile(0.01)
upper = Panel_H3_1['portfolio_vol'].quantile(0.99)
Panel_H3_1['portfolio_vol_wins'] = Panel_H3_1['portfolio_vol'].clip(lower=lower, upper=upper)




Panel_H3_1['log_Negative'] = np.where(Panel_H3_1['Negative'] > 0, np.log(Panel_H3_1['Negative']), 0)
Panel_H3_1['log_Neutral'] = np.where(Panel_H3_1['Neutral'] > 0, np.log(Panel_H3_1['Neutral']), 0)
Panel_H3_1['log_Positive'] = np.where(Panel_H3_1['Positive'] > 0, np.log(Panel_H3_1['Positive']), 0)

Panel_H3_1['log_Negative_weighted'] = np.where(Panel_H3_1['Negative_weighted'] > 0, np.log(Panel_H3_1['Negative_weighted']), 0)
Panel_H3_1['log_Neutral_weighted'] = np.where(Panel_H3_1['Neutral_weighted'] > 0, np.log(Panel_H3_1['Neutral_weighted']), 0)
Panel_H3_1['log_Positive_weighted'] = np.where(Panel_H3_1['Positive_weighted'] > 0, np.log(Panel_H3_1['Positive_weighted']), 0)

Panel_H3_1['log_portfolio_vol'] = np.where(Panel_H3_1['portfolio_vol'] > 0, np.log(Panel_H3_1['portfolio_vol']), 0)
# Panel_H3_1 = Panel_H3_1[Panel_H3_1['portfolio_vol'] != 0]

Panel_H3_1 = Panel_H3_1[Panel_H3_1['log_portfolio_vol'] != 0]
Panel_H3_1['fund_size'] = Panel_H3_1.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

Panel_H3_1['equity_x_netsent']    = Panel_H3_1['is_stocks_fund'] * Panel_H3_1['Net_sent_score_weighted']
Panel_H3_1['bond_x_netsent']    = Panel_H3_1['is_bonds_fund'] * Panel_H3_1['Net_sent_score_weighted']
Panel_H3_1['comb_x_netsent']    = Panel_H3_1['is_comb_fund'] * Panel_H3_1['Net_sent_score_weighted']


In [ ]:
models = run_panel_models(
    df=Panel_H3_1,
    entity_col='fund_key',
    y_var='log_portfolio_vol',
    main_vars=['Net_sent_score_weighted','equity_x_netsent','bond_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/3.vol.1_vs_bonds.xlsx', index=False)

In [ ]:
Panel_H3_2 = pd.merge(panel_df[['date', 'fund_key', 'fund_liquidity']], Panel_agg_ft, on=['date', 'fund_key']).drop_duplicates(subset=['fund_key', 'date'], keep='first').sort_values(by=['date', 'fund_key'], ascending=[False,True])
Panel_H3_2['net_sent_score'] = (Panel_H3_2['Positive'] - Panel_H3_2['Negative'])/(Panel_H3_2['Positive'] + Panel_H3_2['Negative'] + Panel_H3_2['Neutral'])
# Panel_H3_2['is_stocks_fund'] = Panel_H3_2['fund_key'].apply(lambda fk: int(any(name in fk for name in stocks_funds)))
Panel_H3_2=pd.merge(Panel_H3_2, fund_assets_wide.groupby('fund_key').agg({'Категория фонда':'first'}), on = 'fund_key', how = 'left')
Panel_H3_2['Категория фонда'] = Panel_H3_2['Категория фонда'].apply(lambda x: str(x).lower())

Panel_H3_2['is_stocks_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('акции' in x))
Panel_H3_2['is_bonds_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('облигации' in x))
Panel_H3_2['is_comb_fund'] = Panel_H3_2['Категория фонда'].apply(lambda x: int('смешанный' in x))

Panel_H3_2['Neg_share_weighted'] = Panel_H3_2['Negative_weighted'] / (Panel_H3_2['Positive_weighted'] + Panel_H3_2['Negative_weighted'] + Panel_H3_2['Neutral_weighted'])
Panel_H3_2['Pos_share_weighted'] = Panel_H3_2['Positive_weighted'] / (Panel_H3_2['Positive_weighted'] + Panel_H3_2['Negative_weighted'] + Panel_H3_2['Neutral_weighted'])


Panel_H3_2 = Panel_H3_2[(Panel_H3_2['fund_liquidity'] >= Panel_H3_2['fund_liquidity'].quantile(0.01)) &
                    (Panel_H3_2['fund_liquidity'] <= Panel_H3_2['fund_liquidity'].quantile(0.99))]


lower = Panel_H3_2['fund_liquidity'].quantile(0.01)
upper = Panel_H3_2['fund_liquidity'].quantile(0.99)
Panel_H3_2['fund_liquidity_wins'] = Panel_H3_2['fund_liquidity'].clip(lower=lower, upper=upper)



Panel_H3_2['log_Negative'] = np.where(Panel_H3_2['Negative'] > 0, np.log(Panel_H3_2['Negative']), 0)
Panel_H3_2['log_Neutral'] = np.where(Panel_H3_2['Neutral'] > 0, np.log(Panel_H3_2['Neutral']), 0)
Panel_H3_2['log_Positive'] = np.where(Panel_H3_2['Positive'] > 0, np.log(Panel_H3_2['Positive']), 0)

Panel_H3_2['log_Negative_weighted'] = np.where(Panel_H3_2['Negative_weighted'] > 0, np.log(Panel_H3_2['Negative_weighted']), 0)
Panel_H3_2['log_Neutral_weighted'] = np.where(Panel_H3_2['Neutral_weighted'] > 0, np.log(Panel_H3_2['Neutral_weighted']), 0)
Panel_H3_2['log_Positive_weighted'] = np.where(Panel_H3_2['Positive_weighted'] > 0, np.log(Panel_H3_2['Positive_weighted']), 0)

Panel_H3_2['log_fund_liquidity'] = np.where(Panel_H3_2['fund_liquidity'] > 0, np.log(Panel_H3_2['fund_liquidity']), 0)
Panel_H3_2 = Panel_H3_2[Panel_H3_2['fund_liquidity'] != 0]

Panel_H3_2 = Panel_H3_2[Panel_H3_2['log_fund_liquidity'] != 0]
Panel_H3_2['fund_size'] = Panel_H3_2.groupby(['fund_key', 'date'])['Сумма, руб.'].transform('sum')

Panel_H3_2['equity_x_netsent']    = Panel_H3_2['is_stocks_fund'] * Panel_H3_2['Net_sent_score_weighted']
Panel_H3_2['bond_x_netsent']    = Panel_H3_2['is_bonds_fund'] * Panel_H3_2['Net_sent_score_weighted']

In [ ]:
models = run_panel_models(
    df=Panel_H3_2,
    entity_col='fund_key',
    y_var='log_fund_liquidity',
    main_vars=['Net_sent_score_weighted','equity_x_netsent','bond_x_netsent'],
    controls=[
        'Flow','Imoex','imoex_ret',
        'Ruonia','USD_RUB','unique_funds','dummy_mar2014','dummy_feb2022'
    ],
    lag=1,
    entity_effects=False,
    time_effects=False, # also good results if TRUE
    cov_type='clustered',
    cluster_entity=True,
    cluster_time=True
)

print(models)
panel_results_to_df(models).to_excel(r'../../Code/Analisys/regr_res/3.liq.1_vs_bonds.xlsx', index=False)

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1) Copy & prepare your data just like in your run_panel_models
df = Panel_H1.copy()                                 # ← or whichever DataFrame
df['date'] = pd.to_datetime(df['date'])
df = (
    df
    .set_index(['fund_key','date'])
    .sort_index()
    # compute 1-day lag of your main sentiment var
    .assign(net_sent_score_L1=lambda d: d.groupby(level='fund_key')['net_sent_score'].shift(1))
    .dropna(subset=['net_sent_score_L1'])
)

# 2) Specify every regressor in your “full” spec
main_vars = ['net_sent_score']
lagged   = [f"{v}_L1" for v in main_vars]
controls = [
    'Flow','Imoex','imoex_ret','Ruonia','USD_RUB',
    'unique_funds','dummy_mar2014','dummy_feb2022'
]

all_X = main_vars + lagged + controls

# 3) Build the plain-vanilla design matrix for an OLS
X = sm.add_constant(df[all_X])
X = X.replace([np.inf, -np.inf], np.nan).dropna()


# 4) Compute VIFs
vif = pd.DataFrame({
    'variable': X.columns,
    'VIF': [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
})

print(vif)


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# Seed for reproducibility
np.random.seed(0)
plt.rcParams.update({'font.size': 14})
# Generate three hypothetical clusters of mutual funds
cluster_centers = np.array([
    [0.02, 0.02, 0.01],  # low risk, high liquidity, low return
    [0.05, 0.10, 0.04],  # medium risk, medium liquidity, mid return
    [0.08, 0.18, 0.06]   # higher risk, lower liquidity, higher return
])

# Create datapoints around each center
points_per_cluster = 15
data = []
for center in cluster_centers:
    # add some Gaussian noise around each center
    noise = np.random.normal(0, 0.05, size=(points_per_cluster, 3))
    samples = center + noise
    data.append(samples)

data = np.vstack(data)

# Run KMeans clustering
kmeans = KMeans(n_clusters=3, n_init=50, random_state=0)
labels = kmeans.fit_predict(data)

# Plotting
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(data[:,0], data[:,1], data[:,2], c=labels, s=50, edgecolors='k')

ax.set_xlabel('Volatility sensitivity',labelpad=10)
ax.set_ylabel('Liquidity sensitivity',labelpad=10)
ax.set_zlabel('Return')
# ax.set_title('Hypothetical 3‑D Clustering of Mutual Funds')

plt.tight_layout()
plt.show()


In [ ]:
# Panel_agg_it.groupby('date')['share_in_total_assets, %'].sum()

In [ ]:
# import matplotlib.pyplot as plt

# name = 'Сбербанк'

# plt.figure(figsize=(12,6))
# # plt.plot(Panel_agg_it[Panel_agg_it['firm_in_text']== name]['date'], Panel_agg_it[Panel_agg_it['firm_in_text']== name]['share_in_total_assets, %'], marker='o')

# plt.plot(Panel_agg_it.groupby('date')['share_in_total_assets, %'].sum(), marker='o')

# plt.xlabel('Date')
# plt.ylabel('доля')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_agg_it.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# predictors = ['Negative', 'Neutral', 'Positive']
# control_vars = ['Flow', 'Imoex', 'Ruonia'] 

# exog = sm.add_constant(df[predictors + control_vars])

# model = PanelOLS(df['Сумма, руб.'], exog, entity_effects=True, time_effects=False)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)

In [ ]:
# import requests, pandas as pd

# def fetch_spread(engine, market, board, secid, start_date, end_date):
#     dates   = pd.date_range(start_date, end_date)
#     records = []
#     for d in dates:
#         r = requests.get(
#             f"https://iss.moex.com/iss/history/engines/{engine}/markets/{market}"
#             f"/boards/{board}/securities/{secid}/orderbook.json",
#             params={"date": d.strftime("%Y-%m-%d"), "start": 0}
#         ).json()
#         obs = r["history"]["data"]
#         if obs:
#             bid, ask = obs[0][0], obs[0][2]
#             records.append({"date": d, "spread": ask - bid})
#     return pd.DataFrame(records)

# df_spread = fetch_spread(
#     engine   = "stock",
#     market   = "shares",
#     board    = "TQBR",
#     secid    = "SBER",
#     start_date="2024-01-01",
#     end_date  ="2024-04-30"
# )
# print(df_spread.head())
 

## Hypotheses testing

In [ ]:
# import pandas as pd
# import math

# def first_two_nonzero_round(x):
#     """
#     Round x so that the first two non-zero digits after the decimal are kept.
#     """
#     if pd.isna(x):
#         return x
#     absx = abs(x)
#     if absx == 0:
#         return 0.0
#     frac = absx if absx < 1 else absx - math.floor(absx)
#     if frac == 0:
#         decimals = 0
#     else:
#         first_pos = math.ceil(-math.log10(frac))
#         decimals = first_pos + 1
#     return round(x, decimals)

# def results_to_df_two_nonzero(results):
#     """
#     Convert PanelOLS results into a DataFrame with coefficients,
#     standard errors, t-stats, p-values, two-digit dynamic rounding,
#     significance stars, and a combined 'coef_sig' column.
#     """
#     df = pd.DataFrame({
#         'coef':    results.params,
#         'std_err': results.std_errors,
#         't_stat':  results.tstats,
#         'p_value': results.pvalues
#     })
#     # Apply dynamic two-digit rounding
#     for col in ['coef', 'std_err', 't_stat', 'p_value']:
#         df[col] = df[col].apply(first_two_nonzero_round)
#     # Add significance stars
#     def stars(p):
#         if p < 0.01: return '***'
#         elif p < 0.05: return '**'
#         elif p < 0.10: return '*'
#         return ''
#     df['signif'] = df['p_value'].apply(stars)
#     # Create combined coefficient + stars column
#     df['coef_sig'] = df['coef'].astype(str) + df['signif']
#     return df

# # Example usage:
# # results_df = results_to_df_two_nonzero(results)
# # ace_tools.display_dataframe_to_user("FE Results with Coef+Stars", results_df)


<!-- ## H1 - weights std -->

In [ ]:
# Panel_H1 = Panel_agg_fit.drop_duplicates(subset=['firm_in_text', 'date'], keep='first').drop(['fund_key','Класс активов'], axis=1)
# Panel_H1['news_count'] = Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Neutral']
# Panel_H1['net_sent_score'] = (Panel_H1['Positive'] - Panel_H1['Negative'])/(Panel_H1['Positive'] + Panel_H1['Negative'] + Panel_H1['Positive'])

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# i = 1
# df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})


# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['weights_std']
# X = ['news_count']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB','unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=True, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# i = 1
# df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})


# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['weights_std']
# X = ['Negative', 'Neutral', 'Positive']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB','unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=True, time_effects=True, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# Y = df['weights_std']
# X = ['Negative', 'Neutral', 'Positive']
# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB'] 

# exog = sm.add_constant(df[X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=True, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)

In [ ]:
# 

In [ ]:
# import pandas as pd
# import numpy as np
# import statsmodels.api as sm
# from scipy import stats
# from linearmodels.panel import PanelOLS, RandomEffects

# # 1. Prepare the panel DataFrame
# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# # 2. Dependent & independent variables
# Y = df['weights_std']
# X = ['news_count']
# controls = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB']
# exog = sm.add_constant(df[X + controls])

# # 3. Fit FE model
# fe_mod = PanelOLS(Y, exog, entity_effects=True, time_effects=False, drop_absorbed=True)
# fe_res = fe_mod.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
# print("===== FE Results =====")
# print(fe_res.summary)

# # 4. Fit RE model
# re_mod = RandomEffects(Y, exog)
# re_res = re_mod.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
# print("\n===== RE Results =====")
# print(re_res.summary)

# # 5. Manual Hausman test
# # Extract parameter vectors
# beta_fe = fe_res.params
# beta_re = re_res.params

# # Align indices
# common_idx = beta_fe.index.intersection(beta_re.index)
# d = beta_fe[common_idx] - beta_re[common_idx]

# # Extract covariance matrices
# cov_fe = fe_res.cov.loc[common_idx, common_idx]
# cov_re = re_res.cov.loc[common_idx, common_idx]

# # Compute difference of covariances
# V_diff = cov_fe - cov_re

# # Compute the test statistic
# H = float(d.T @ np.linalg.inv(V_diff.values) @ d)
# df_chi2 = len(d)  # degrees of freedom = number of parameters compared
# p_value = 1 - stats.chi2.cdf(H, df_chi2)

# print("\n===== Hausman Test =====")
# print(f"χ²({df_chi2}) = {H:.2f}")
# print(f"p-value    = {p_value:.4f}")

# if p_value < 0.05:
#     print("→ Reject H0: RE is inconsistent; FE preferred.")
# else:
#     print("→ Cannot reject H0: RE may be used (more efficient).")


In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import RandomEffects

# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# Y = df['weights_std']
# X = ['news_count']
# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB'] 

# exog = sm.add_constant(df[X + control_vars])

# model = RandomEffects(Y, exog)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# # 1. Load & index your data
# df = Panel_H1.copy()
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['firm_in_text', 'date'])

# # 2. Define Y, X, controls, exog
# Y = df['weights_std']
# X = ['news_count']
# controls = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB']
# exog = sm.add_constant(df[X + controls])

# # 3. Pooled OLS (no FE)
# pooled_res = PanelOLS(Y, exog,
#                       entity_effects=False,
#                       time_effects=False,
#                       drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
# print("===== Pooled Model =====")
# print(pooled_res.summary)

# # 4. Fixed‐Effects model
# fe_res = PanelOLS(Y, exog,
#                   entity_effects=True,
#                   time_effects=False,
#                   drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
# print("\n===== Fixed‐Effects Model =====")
# print(fe_res.summary)

# # 5. Redundant‐FE F‐test
# f_test = fe_res.f_statistic

# print("\n===== Pooled vs FE F‐Test =====")
# # Just print the F‐statistic and p‐value
# print(f"F = {f_test.stat:.3f}, p‐value = {f_test.pval:.3f}")

# if f_test.pval < 0.05:
#     print("→ Reject H₀: entity (firm) effects are jointly significant. Use FE.")
# else:
#     print("→ Cannot reject H₀: pooled OLS may suffice.")


In [ ]:
# 

<!-- ## H2 - Assets classes -->

In [ ]:
# import matplotlib.pyplot as plt

# # Compute the number of unique fund_key values for each date
# unique_funds_by_date = Panel.groupby("date")["fund_key"].nunique().reset_index(name="unique_funds")

# # Plot a histogram of the unique funds per date
# plt.figure(figsize=(10, 6))
# plt.hist(unique_funds_by_date["unique_funds"], bins=20, edgecolor="black", color="skyblue")
# plt.xlabel("Number of Unique Funds")
# plt.ylabel("Frequency")
# plt.title("Histogram of Unique Funds Count per Date")
# plt.show()

In [ ]:
# Panel_H2 = Panel_agg_ct.sort_values(by=['date','Класс активов'], ascending=[False, True])
# Panel_H2['news_count'] = Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Neutral']
# Panel_H2['net_sent_score'] = (Panel_H2['Positive'] - Panel_H2['Negative'])/(Panel_H2['Positive'] + Panel_H2['Negative'] + Panel_H2['Positive'])

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # Define list of dates to drop
# dates_to_drop = [
#     "2021-08-31", "2021-07-30", "2021-05-31", "2021-04-30",
#     "2018-04-28", "2018-05-31", "2019-07-31", "2019-08-30",
#     "2019-01-31", "2020-01-31", "2019-05-31", "2019-10-31",
#     "2019-04-30", "2018-07-31", "2018-11-30", "2019-02-28",
#     "2018-10-31", "2019-11-29", "2021-02-26", "2020-05-29",
#     "2020-07-31", "2020-02-28", "2020-08-31", "2021-01-29",
#     "2020-04-30", "2020-10-30", "2020-11-30", "2018-08-31"
# ]
# dates_to_drop = pd.to_datetime(dates_to_drop)

# # Filter data for bonds and russian_stocks in 'Класс активов'
# df_bonds = Panel_H2[Panel_H2["Класс активов"] == "bonds"].sort_values("date")
# df_russian = Panel_H2[Panel_H2["Класс активов"] == "russian_stocks"].sort_values("date")

# # Create filtered copies without altering the original DataFrames
# df_bonds_filtered = df_bonds[~df_bonds["date"].isin(dates_to_drop)]
# df_russian_filtered = df_russian[~df_russian["date"].isin(dates_to_drop)]


# # Plot bonds and russian_stocks shares on primary y-axis
# fig, ax = plt.subplots(figsize=(12, 6))
# ax.plot(df_bonds_filtered["date"], df_bonds_filtered["assets_class_share"], label="Bonds", marker="o", linewidth=2)
# ax.plot(df_russian_filtered["date"], df_russian_filtered["assets_class_share"], label="Russian Stocks", marker="o", linewidth=2)
# ax.set_xlabel("Date")
# ax.set_ylabel("Assets Class Share")
# ax.set_title("Assets Class Share & Net Sentiment Over Time (Dropped Selected Dates)")

# # Plot net_sent_score (in %) on secondary y-axis
# ax2 = ax.twinx()
# ax2.plot(df_bonds_filtered["date"], df_bonds_filtered["net_sent_score"], label="Net Sentiment Score", color="blue", marker="o", linewidth=2)
# ax2.set_ylabel("Net Sentiment (%)")

# # Combine legends from both axes
# lines1, labels1 = ax.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# plt.show()

In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'bonds']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})
# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['share_chng']
# X = ['net_sent_score']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB', 'unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# # results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'bonds']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('firm_in_text')['news_count'].shift(lag)})
# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['share_chng']
# X = ['net_sent_score']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# # L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB','unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# # results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'russian_stocks']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'net_sent_score{i}': lambda d, lag=i: d.groupby('Класс активов')['net_sent_score'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'net_sent_score{i}': lambda d, lag=i: d.groupby('firm_in_text')['net_sent_score'].shift(lag)})


# df.dropna(subset=[f'net_sent_score{i}'], inplace=True)

# Y = df['share_chng']
# X = ['net_sent_score']
# L_X = [
#     # f'net_sent_score{i-1}',
#     f'net_sent_score{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'russian_stocks']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'net_sent_score{i}': lambda d, lag=i: d.groupby('Класс активов')['net_sent_score'].shift(lag)})
# # i = 2
# # df = df.sort_values(['firm_in_text', 'date']).assign(**{f'net_sent_score{i}': lambda d, lag=i: d.groupby('firm_in_text')['net_sent_score'].shift(lag)})


# df.dropna(subset=[f'net_sent_score{i}'], inplace=True)

# Y = df['share_chng']
# X = ['net_sent_score']
# L_X = [
#     # f'net_sent_score{i-1}',
#     f'net_sent_score{i}']

# # L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# # results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# 

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'bonds']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})


# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['share_chng']
# X = ['Negative', 'Neutral', 'Positive']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB','unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# import pandas as pd
# import statsmodels.api as sm
# from linearmodels.panel import PanelOLS

# df = Panel_H2.copy()
# df = df[df['Класс активов'] == 'russian_stocks']

# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index(['Класс активов', 'date'])

# i = 1
# df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})
# # i = 2
# # df = df.sort_values(['Класс активов', 'date']).assign(**{f'news_count_lag{i}': lambda d, lag=i: d.groupby('Класс активов')['news_count'].shift(lag)})


# df.dropna(subset=[f'news_count_lag{i}'], inplace=True)

# Y = df['share_chng']
# X = ['Negative', 'Neutral', 'Positive']
# L_X = [
#     # f'news_count_lag{i-1}',
#     f'news_count_lag{i}']

# L_X = []

# control_vars = ['Flow', 'Imoex', 'imoex_ret', 'Ruonia', 'USD_RUB','unique_funds', 'dummy_mar2014', 'dummy_feb2022'] 
# # control_vars = []

# exog = sm.add_constant(df[X + L_X + control_vars])

# model = PanelOLS(Y, exog, entity_effects=False, time_effects=False, drop_absorbed=True)
# results = model.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

# print(results)
# results_df = results_to_df_two_nonzero(results)
# results_df.to_excel("FE_REGR_output.xlsx", index=True, engine="openpyxl")

In [ ]:
# 